In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION


Purpose:
    Develop the final multi-label disease classification head using the
    locked Experiment E (Seed 42) bilateral fused representation.

Locked Upstream Configuration:
    • Experiment: E — Adaptive Bilateral Fusion
    • Seed: 42
    • Best Epoch: 5
    • Val Macro-F1: 0.561936
    • Val Micro-F1: 0.535966
    • Exact Match: 0.175799
    • Locked artifact: stage2_experiment_e_locked_seed42.pt

Strict Constraints:
    • Experiment E is FROZEN.
    • Cross-Eye representations are FROZEN.
    • Adaptive bilateral fusion is FROZEN.
    • No changes to the upstream backbone, disease attention,
      cross-eye attention, or fusion mechanism.
    • Only the downstream multi-label classification module will be optimized.

Classification Objective:
    Predict the 8 disease labels simultaneously from the locked fused
    bilateral representation.

Optimization Goal:
    Maximize multi-label diagnostic performance, with particular emphasis on:
    • Macro-F1
    • Micro-F1
    • Per-disease F1
    • Exact Match
    • Robust handling of class imbalance

Experimental Strategy:
    Start with strong, established multi-label classification approaches
    rather than weak baseline architectures or arbitrary architectural changes.

    Initial focus:
    • Strong classification head
    • Appropriate normalization and regularization
    • Class-imbalance-aware loss
    • Multi-label-specific loss functions
    • Validation-based threshold optimization
    • Per-disease performance analysis

Important:
    The classification module must consume the locked Experiment E
    representation directly. Upstream representations must not be retrained
    or modified during classifier development.



In [3]:
# =============================================================================
# QCDP-BiFormer — MODULE 9
# MULTI-LABEL CLASSIFICATION
# CELL 1 — ENVIRONMENT + LOCKED RESOURCES
# =============================================================================

import os
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive

drive.mount('/content/drive')

ROOT = Path("/content/drive/My Drive/Eye Disease/Dataset")

assert ROOT.exists(), f"Dataset root not found: {ROOT}"

print("=" * 80)
print("QCDP-BiFormer — MULTI-LABEL CLASSIFICATION")
print("=" * 80)
print(f"Dataset root: {ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior for evaluation/reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"\nGlobal seed: {SEED}")

# -----------------------------------------------------------------------------
# 3. LOCKED EXPERIMENT
# -----------------------------------------------------------------------------

LOCKED_EXPERIMENT = "experiment_e"
LOCKED_SEED = 42

LOCKED_ARTIFACT = ROOT / "stage2_experiment_e_locked_seed42.pt"

assert LOCKED_ARTIFACT.exists(), (
    f"Locked Experiment E artifact not found:\n{LOCKED_ARTIFACT}"
)

print("\nLocked configuration:")
print(f"  Experiment : {LOCKED_EXPERIMENT}")
print(f"  Seed       : {LOCKED_SEED}")
print(f"  Artifact   : {LOCKED_ARTIFACT}")

# -----------------------------------------------------------------------------
# 4. DATASET / LABEL DEFINITIONS
# -----------------------------------------------------------------------------

TRAIN_DF_PATH = ROOT / "stage2_train_df.csv"
VAL_DF_PATH   = ROOT / "val_patient_df.csv"

assert TRAIN_DF_PATH.exists(), f"Missing: {TRAIN_DF_PATH}"
assert VAL_DF_PATH.exists(), f"Missing: {VAL_DF_PATH}"

train_df = pd.read_csv(TRAIN_DF_PATH)
val_df   = pd.read_csv(VAL_DF_PATH)

# Fixed disease ordering used throughout QCDP-BiFormer
DISEASE_NAMES = [
    "N",  # Normal
    "D",  # Diabetic Retinopathy
    "G",  # Glaucoma
    "C",  # Cataract
    "A",  # AMD
    "H",  # Hypertension
    "M",  # Myopia
    "O",  # Other
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\nDataset:")
print(f"  Train samples : {len(train_df)}")
print(f"  Val samples   : {len(val_df)}")
print(f"  Classes       : {NUM_CLASSES}")
print(f"  Label order   : {DISEASE_NAMES}")

# -----------------------------------------------------------------------------
# 5. EXISTING LOCKED / UPSTREAM RESOURCES
# -----------------------------------------------------------------------------

RESOURCE_FILES = {
    "stage1_train_df": ROOT / "stage1_train_df.csv",
    "stage2_train_df": ROOT / "stage2_train_df.csv",
    "stage1_quality_scores": ROOT / "stage1_quality_scores.csv",

    "disease_aware_features": ROOT / "disease_aware_features.pt",
    "disease_attention_weights": ROOT / "disease_attention_weights.pt",
    "disease_prototypes": ROOT / "disease_prototypes.pt",

    "cross_eye_train": ROOT / "stage2_train_cross_eye_features_final.pt",
    "cross_eye_val": ROOT / "stage2_val_cross_eye_features_final.pt",

    "locked_experiment_e": LOCKED_ARTIFACT,
}

print("\nUpstream resources:")
for name, path in RESOURCE_FILES.items():
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"  [{status:>9}] {name}: {path.name}")

# -----------------------------------------------------------------------------
# 6. LOAD LOCKED EXPERIMENT E ARTIFACT
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("Loading locked Experiment E artifact...")
print("-" * 80)

locked_artifact = torch.load(
    LOCKED_ARTIFACT,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(locked_artifact)}")

# Inspect structure without modifying anything
if isinstance(locked_artifact, dict):

    print("\nArtifact contents:")
    for key, value in locked_artifact.items():

        if torch.is_tensor(value):
            print(
                f"  {key}: Tensor "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )

        elif isinstance(value, dict):
            print(
                f"  {key}: dict "
                f"({len(value)} entries)"
            )

        elif isinstance(value, (list, tuple)):
            print(
                f"  {key}: {type(value).__name__} "
                f"({len(value)} entries)"
            )

        else:
            print(
                f"  {key}: "
                f"{type(value).__name__} = {value}"
            )

# -----------------------------------------------------------------------------
# 7. LOCKED REFERENCE METRICS
# -----------------------------------------------------------------------------

LOCKED_REFERENCE = {
    "best_epoch": 5,
    "val_loss": 0.791240,
    "micro_f1": 0.535966,
    "macro_f1": 0.561936,
    "exact_match": 0.175799,
}

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E / SEED 42 REFERENCE")
print("-" * 80)

for metric, value in LOCKED_REFERENCE.items():
    print(f"{metric:>15}: {value}")

# -----------------------------------------------------------------------------
# 8. GLOBAL CLASSIFICATION CONFIGURATION
# -----------------------------------------------------------------------------

FEATURE_DIM = 768
NUM_CLASSES = 8

# We will NOT modify these upstream representations.
UPSTREAM_FROZEN = True

print("\n" + "=" * 80)
print("MODULE 9 INITIALIZATION COMPLETE")
print("=" * 80)

print(f"""
Locked Experiment       : E
Locked Seed             : 42
Feature dimension       : {FEATURE_DIM}
Number of diseases      : {NUM_CLASSES}
Disease ordering        : {DISEASE_NAMES}

Upstream fusion         : FROZEN
Cross-Eye representations: FROZEN
Adaptive fusion         : FROZEN
Mean-fusion replacement : NOT ALLOWED

""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QCDP-BiFormer — MULTI-LABEL CLASSIFICATION
Dataset root: /content/drive/My Drive/Eye Disease/Dataset
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Global seed: 42

Locked configuration:
  Experiment : experiment_e
  Seed       : 42
  Artifact   : /content/drive/My Drive/Eye Disease/Dataset/stage2_experiment_e_locked_seed42.pt

Dataset:
  Train samples : 2141
  Val samples   : 504
  Classes       : 8
  Label order   : ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Upstream resources:
  [    FOUND] stage1_train_df: stage1_train_df.csv
  [    FOUND] stage2_train_df: stage2_train_df.csv
  [    FOUND] stage1_quality_scores: stage1_quality_scores.csv
  [    FOUND] disease_aware_features: disease_aware_features.pt
  [    FOUND] disease_attention_weights: disease_attention_weights.pt
  [    FOUND] disease_prototypes: disease_prototypes.pt
  [    FOUND]

CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS


Objective:
    Load the officially exported Experiment E / Seed 42 fused representations
    and verify their integrity before designing and training the multi-label
    classification head.

OFFICIALLY LOCKED UPSTREAM:
    Experiment E — Strict Class-wise Adaptive Fusion
    Seed: 42

Representation:
    Train: [2141, 8, 768]
    Val  : [438, 8, 768]

Interpretation:
    Each patient has 8 disease-specific fused representations, with each
    disease receiving its own 768-dimensional representation.

Disease order:
    ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Disease mapping:
    N = Normal
    D = Diabetic Retinopathy
    G = Glaucoma
    C = Cataract
    A = Age-related Macular Degeneration
    H = Hypertension
    M = Myopia
    O = Other

Validation cohort:
    The 438-patient bilateral cohort is used because it is the exact cohort
    on which the locked Experiment E representation was generated.

STRICT UPSTREAM LOCK:
    • No fusion retraining
    • No feature modification
    • No new patient filtering
    • No new train/validation split
    • No mean-fusion replacement
    • No changes to Experiment E

This cell performs verification only.

Next:
    Determine the most appropriate classification-head input strategy for
    the disease-specific [8 × 768] representation before beginning
    optimization experiments.


In [4]:
# =============================================================================
# QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION
# CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("=" * 80)
print("CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/My Drive/Eye Disease/Dataset"
)

FUSED_ARTIFACT_PATH = (
    ROOT / "stage2_experiment_e_fused_representations_seed42.pt"
)

TRAIN_LABEL_PATH = (
    ROOT / "stage2_train_bilateral_metadata.csv"
)

VAL_LABEL_PATH = (
    ROOT / "stage2_val_bilateral_metadata.csv"
)

assert FUSED_ARTIFACT_PATH.exists(), (
    f"Missing fused representation artifact:\n{FUSED_ARTIFACT_PATH}"
)

assert TRAIN_LABEL_PATH.exists(), (
    f"Missing train bilateral metadata:\n{TRAIN_LABEL_PATH}"
)

assert VAL_LABEL_PATH.exists(), (
    f"Missing validation bilateral metadata:\n{VAL_LABEL_PATH}"
)

print(f"\nDataset root: {ROOT}")
print(f"Fused artifact: {FUSED_ARTIFACT_PATH.name}")


# =============================================================================
# 2. LOAD ARTIFACT
# =============================================================================

print("\n" + "-" * 80)
print("LOADING LOCKED FUSED REPRESENTATIONS")
print("-" * 80)

fused_artifact = torch.load(
    FUSED_ARTIFACT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(fused_artifact)}")


# =============================================================================
# 3. VERIFY LOCKED PROVENANCE
# =============================================================================

print("\n" + "-" * 80)
print("VERIFYING LOCKED PROVENANCE")
print("-" * 80)

print(
    f"Experiment     : "
    f"{fused_artifact.get('experiment')}"
)

print(
    f"Experiment key : "
    f"{fused_artifact.get('experiment_key')}"
)

print(
    f"Seed           : "
    f"{fused_artifact.get('seed')}"
)

print(
    f"Status         : "
    f"{fused_artifact.get('status')}"
)

assert fused_artifact["experiment_key"] == "experiment_e"
assert fused_artifact["seed"] == 42
assert fused_artifact["status"] == "LOCKED"

print("\n✓ Experiment E verified.")
print("✓ Seed 42 verified.")
print("✓ Artifact status = LOCKED.")


# =============================================================================
# 4. DISEASE ORDER
# =============================================================================

DISEASE_NAMES = [
    "N",
    "D",
    "G",
    "C",
    "A",
    "H",
    "M",
    "O",
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\n" + "-" * 80)
print("DISEASE ORDER")
print("-" * 80)

print(DISEASE_NAMES)

artifact_labels = list(
    fused_artifact["label_columns"]
)

print(
    f"\nArtifact label order: {artifact_labels}"
)

assert artifact_labels == DISEASE_NAMES, (
    "Artifact disease ordering does not match Module 9 ordering."
)

print("✓ Disease ordering verified.")


# =============================================================================
# 5. EXTRACT FUSED FEATURES
# =============================================================================

print("\n" + "-" * 80)
print("EXTRACTING FUSED FEATURES")
print("-" * 80)

X_train = fused_artifact[
    "train_classwise_features"
].float()

X_val = fused_artifact[
    "val_classwise_features"
].float()


# -----------------------------------------------------------------------------
# IMPORTANT:
# The fused artifact stores patient IDs as 0-D PyTorch tensors:
#     tensor(1), tensor(4), ...
#
# The CSV stores them as integers:
#     1, 4, ...
#
# Normalize them here so the remainder of the notebook works with ordinary
# Python integer patient IDs.
# -----------------------------------------------------------------------------

def normalize_patient_ids(ids):

    normalized = []

    for x in ids:

        if torch.is_tensor(x):
            normalized.append(int(x.item()))

        else:
            normalized.append(int(x))

    return normalized


train_patient_ids = normalize_patient_ids(
    fused_artifact["train_patient_ids"]
)

val_patient_ids = normalize_patient_ids(
    fused_artifact["val_patient_ids"]
)

print(
    f"Train fused representation : "
    f"{tuple(X_train.shape)}"
)

print(
    f"Val fused representation   : "
    f"{tuple(X_val.shape)}"
)

print(
    f"Train patient IDs          : "
    f"{len(train_patient_ids)}"
)

print(
    f"Val patient IDs            : "
    f"{len(val_patient_ids)}"
)

print(
    f"\nFirst 10 normalized train IDs:"
)

print(train_patient_ids[:10])


# =============================================================================
# 6. STRICT SHAPE VERIFICATION
# =============================================================================

assert X_train.ndim == 3
assert X_val.ndim == 3

assert X_train.shape[1] == NUM_CLASSES
assert X_val.shape[1] == NUM_CLASSES

assert X_train.shape[2] == 768
assert X_val.shape[2] == 768

assert X_train.shape[0] == len(train_patient_ids)
assert X_val.shape[0] == len(val_patient_ids)

print("\n✓ Train shape = [2141, 8, 768]")
print("✓ Validation shape = [438, 8, 768]")
print("✓ Eight disease-specific representations confirmed.")
print("✓ Representation dimension = 768.")


# =============================================================================
# 7. CHECK FINITE VALUES
# =============================================================================

print("\n" + "-" * 80)
print("NUMERICAL INTEGRITY")
print("-" * 80)

train_nan = torch.isnan(X_train).sum().item()
train_inf = torch.isinf(X_train).sum().item()

val_nan = torch.isnan(X_val).sum().item()
val_inf = torch.isinf(X_val).sum().item()

print(f"Train NaN values : {train_nan}")
print(f"Train Inf values : {train_inf}")
print(f"Val NaN values   : {val_nan}")
print(f"Val Inf values   : {val_inf}")

assert train_nan == 0
assert train_inf == 0
assert val_nan == 0
assert val_inf == 0

print("\n✓ No NaN values.")
print("✓ No infinite values.")


# =============================================================================
# 8. REPRESENTATION STATISTICS
# =============================================================================

print("\n" + "-" * 80)
print("REPRESENTATION STATISTICS")
print("-" * 80)

print(
    f"Train mean : {X_train.mean().item():.6f}"
)

print(
    f"Train std  : {X_train.std().item():.6f}"
)

print(
    f"Train min  : {X_train.min().item():.6f}"
)

print(
    f"Train max  : {X_train.max().item():.6f}"
)

print()

print(
    f"Val mean   : {X_val.mean().item():.6f}"
)

print(
    f"Val std    : {X_val.std().item():.6f}"
)

print(
    f"Val min    : {X_val.min().item():.6f}"
)

print(
    f"Val max    : {X_val.max().item():.6f}"
)


# =============================================================================
# 9. LOAD BILATERAL LABEL DATA
# =============================================================================

print("\n" + "-" * 80)
print("LOADING BILATERAL LABEL DATA")
print("-" * 80)

train_df = pd.read_csv(
    TRAIN_LABEL_PATH
)

val_df = pd.read_csv(
    VAL_LABEL_PATH
)

print(
    f"Train dataframe: {train_df.shape}"
)

print(
    f"Validation dataframe: {val_df.shape}"
)


# =============================================================================
# 10. VERIFY LABEL COLUMNS
# =============================================================================

missing_train = [
    disease for disease in DISEASE_NAMES
    if disease not in train_df.columns
]

missing_val = [
    disease for disease in DISEASE_NAMES
    if disease not in val_df.columns
]

assert not missing_train, (
    f"Missing train labels: {missing_train}"
)

assert not missing_val, (
    f"Missing validation labels: {missing_val}"
)

print("\n✓ All eight disease labels present.")


# =============================================================================
# 11. VERIFY PATIENT COUNTS
# =============================================================================

assert len(train_df) == X_train.shape[0]
assert len(val_df) == X_val.shape[0]

print("\n" + "-" * 80)
print("PATIENT COUNT ALIGNMENT")
print("-" * 80)

print(
    f"Train features : {len(X_train)}"
)

print(
    f"Train labels   : {len(train_df)}"
)

print(
    f"Val features   : {len(X_val)}"
)

print(
    f"Val labels     : {len(val_df)}"
)

print("\n✓ Train counts aligned.")
print("✓ Validation counts aligned.")


# =============================================================================
# 12. VERIFY PATIENT-ID ALIGNMENT
# =============================================================================

print("\n" + "-" * 80)
print("PATIENT-ID ALIGNMENT")
print("-" * 80)

possible_id_columns = [
    "patient_id",
    "Patient_ID",
    "patientID",
    "PatientID",
    "patient",
    "ID",
    "id",
]

train_id_candidates = [
    c for c in possible_id_columns
    if c in train_df.columns
]

val_id_candidates = [
    c for c in possible_id_columns
    if c in val_df.columns
]

print(
    f"Train ID candidates: {train_id_candidates}"
)

print(
    f"Val ID candidates  : {val_id_candidates}"
)

assert len(train_id_candidates) == 1, (
    "Could not uniquely identify train patient-ID column."
)

assert len(val_id_candidates) == 1, (
    "Could not uniquely identify validation patient-ID column."
)

train_id_col = train_id_candidates[0]
val_id_col = val_id_candidates[0]

train_df_ids = [
    int(x)
    for x in train_df[train_id_col].tolist()
]

val_df_ids = [
    int(x)
    for x in val_df[val_id_col].tolist()
]

assert len(train_df_ids) == len(set(train_df_ids))
assert len(val_df_ids) == len(set(val_df_ids))

train_feature_set = set(train_patient_ids)
train_label_set = set(train_df_ids)

val_feature_set = set(val_patient_ids)
val_label_set = set(val_df_ids)

assert train_feature_set == train_label_set, (
    "Train patient IDs do not match exactly."
)

assert val_feature_set == val_label_set, (
    "Validation patient IDs do not match exactly."
)

print("\n✓ Train patient IDs match exactly.")
print("✓ Validation patient IDs match exactly.")


# =============================================================================
# 13. BUILD LABEL MATRICES
# =============================================================================

print("\n" + "-" * 80)
print("BUILDING MULTI-LABEL TARGET MATRICES")
print("-" * 80)

# Reorder dataframes explicitly according to the fused-feature patient order.

train_lookup = train_df.set_index(train_id_col)

val_lookup = val_df.set_index(val_id_col)

train_ordered_df = train_lookup.loc[
    train_patient_ids
].reset_index()

val_ordered_df = val_lookup.loc[
    val_patient_ids
].reset_index()

Y_train = torch.tensor(
    train_ordered_df[DISEASE_NAMES].values,
    dtype=torch.float32
)

Y_val = torch.tensor(
    val_ordered_df[DISEASE_NAMES].values,
    dtype=torch.float32
)

print(
    f"Y_train shape: {tuple(Y_train.shape)}"
)

print(
    f"Y_val shape  : {tuple(Y_val.shape)}"
)

assert Y_train.shape == (
    X_train.shape[0],
    NUM_CLASSES
)

assert Y_val.shape == (
    X_val.shape[0],
    NUM_CLASSES
)

print("\n✓ Feature → patient → label ordering verified.")


# =============================================================================
# 14. VERIFY BINARY LABELS
# =============================================================================

unique_train_labels = torch.unique(Y_train).tolist()
unique_val_labels = torch.unique(Y_val).tolist()

print("\nTrain unique label values:", unique_train_labels)
print("Val unique label values  :", unique_val_labels)

assert set(unique_train_labels).issubset({0.0, 1.0})
assert set(unique_val_labels).issubset({0.0, 1.0})

print("\n✓ Targets are binary multi-label indicators.")


# =============================================================================
# 15. CLASS DISTRIBUTION
# =============================================================================

print("\n" + "-" * 80)
print("CLASS DISTRIBUTION")
print("-" * 80)

print(
    f"{'Disease':>8} | "
    f"{'Train +':>8} | "
    f"{'Train %':>8} | "
    f"{'Val +':>8} | "
    f"{'Val %':>8}"
)

print("-" * 55)

for i, disease in enumerate(DISEASE_NAMES):

    train_pos = int(Y_train[:, i].sum().item())
    val_pos = int(Y_val[:, i].sum().item())

    train_prev = train_pos / len(Y_train)
    val_prev = val_pos / len(Y_val)

    print(
        f"{disease:>8} | "
        f"{train_pos:8d} | "
        f"{train_prev:8.4f} | "
        f"{val_pos:8d} | "
        f"{val_prev:8.4f}"
    )


# =============================================================================
# 16. MULTI-LABEL CARDINALITY
# =============================================================================

train_cardinality = Y_train.sum(dim=1)
val_cardinality = Y_val.sum(dim=1)

print("\n" + "-" * 80)
print("MULTI-LABEL CARDINALITY")
print("-" * 80)

print(
    f"Train mean labels/patient : "
    f"{train_cardinality.mean().item():.4f}"
)

print(
    f"Val mean labels/patient   : "
    f"{val_cardinality.mean().item():.4f}"
)

print(
    f"Train max labels/patient  : "
    f"{int(train_cardinality.max().item())}"
)

print(
    f"Val max labels/patient    : "
    f"{int(val_cardinality.max().item())}"
)


# =============================================================================
# 17. VERIFY LOCKED REFERENCE
# =============================================================================

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E REFERENCE")
print("-" * 80)

locked_reference = fused_artifact[
    "locked_reference"
]

for key, value in locked_reference.items():
    print(
        f"{key:20s}: {value}"
    )

assert locked_reference["best_epoch"] == 5

assert abs(
    locked_reference["val_micro_f1"] - 0.535966
) < 1e-6

assert abs(
    locked_reference["val_macro_f1"] - 0.561936
) < 1e-6

assert abs(
    locked_reference["exact_match"] - 0.175799
) < 1e-6

print("\n✓ Locked Experiment E reference verified.")





CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS

Dataset root: /content/drive/My Drive/Eye Disease/Dataset
Fused artifact: stage2_experiment_e_fused_representations_seed42.pt

--------------------------------------------------------------------------------
LOADING LOCKED FUSED REPRESENTATIONS
--------------------------------------------------------------------------------
Artifact loaded successfully.
Artifact type: <class 'dict'>

--------------------------------------------------------------------------------
VERIFYING LOCKED PROVENANCE
--------------------------------------------------------------------------------
Experiment     : Experiment E — Strict Class-wise Adaptive Fusion
Experiment key : experiment_e
Seed           : 42
Status         : LOCKED

✓ Experiment E verified.
✓ Seed 42 verified.
✓ Artifact status = LOCKED.

--------------------------------------------------------------------------------
DISEASE ORDER
-----------------------------------------------------

###EXPERIMENT 1


UPSTREAM HANDOFF VERIFIED

Train:

    X_train : [2141, 8, 768]
    Y_train : [2141, 8]

Validation:

    X_val   : [438, 8, 768]
    Y_val   : [438, 8]

Representation:

    Eight disease-specific 768-D fused representations per patient.

Patient alignment:

    ✓ Verified
    ✓ Tensor IDs normalized to integer IDs

Labels:

    ✓ Binary multi-label
    ✓ Correct disease ordering

Experiment E:

    ✓ Locked
    ✓ Seed 42
    ✓ No upstream modification

IMPORTANT:

    No classifier has been trained yet.

The next step is architectural design of the multi-label classification
head using the locked disease-specific fused representations.



CELL 3 : REPRESENTATION-AWARE CLASSIFICATION HEAD


 Input:

       [B, 8, 768]

 Each of the 8 slots corresponds to one disease-specific representation.

 Architecture:

       Disease-specific fused features
                  [B, 8, 768]
                         │
                    LayerNorm
                         │
                Shared projection
                  768 → 512
                         │
                       ReLU
                         │
                     Dropout
                         │
                Disease-specific
                   scalar heads
                         │
                  [B, 8, 1]
                         │
                       logits

 The shared projection learns a common representation space while the
 disease-specific heads preserve independent classification decisions.


In [5]:
# =============================================================================
# QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION
# CELL 3 — REPRESENTATION-AWARE CLASSIFICATION HEAD
# =============================================================================

import torch
import torch.nn as nn

print("=" * 80)
print("CELL 3 — REPRESENTATION-AWARE MULTI-LABEL CLASSIFICATION HEAD")
print("=" * 80)


# =============================================================================
# 1. LOCKED INPUT CONFIGURATION
# =============================================================================

NUM_CLASSES = 8
INPUT_DIM = 768
HIDDEN_DIM = 512
DROPOUT = 0.30

print("\n" + "-" * 80)
print("LOCKED INPUT CONFIGURATION")
print("-" * 80)

print(f"Number of disease representations : {NUM_CLASSES}")
print(f"Representation dimension          : {INPUT_DIM}")
print(f"Hidden dimension                  : {HIDDEN_DIM}")
print(f"Dropout                           : {DROPOUT}")


# =============================================================================
# 2. REPRESENTATION-AWARE CLASSIFICATION HEAD
# =============================================================================

class RepresentationAwareMultiLabelHead(nn.Module):

    def __init__(
        self,
        input_dim=768,
        hidden_dim=512,
        num_classes=8,
        dropout=0.30
    ):

        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes

        # ---------------------------------------------------------------------
        # Per-disease normalization
        # ---------------------------------------------------------------------

        self.layer_norm = nn.LayerNorm(
            input_dim
        )

        # ---------------------------------------------------------------------
        # Shared representation projection
        # ---------------------------------------------------------------------

        self.shared_projection = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        # ---------------------------------------------------------------------
        # Independent disease classifiers
        #
        # Each disease receives its own classifier.
        # This is important because the eight diseases have very different
        # prevalence and visual characteristics.
        # ---------------------------------------------------------------------

        self.disease_heads = nn.ModuleList([

            nn.Linear(
                hidden_dim,
                1
            )

            for _ in range(num_classes)

        ])


    def forward(self, x):

        # Expected:
        # [batch, 8, 768]

        assert x.ndim == 3, (
            f"Expected [B, 8, 768], got {tuple(x.shape)}"
        )

        assert x.shape[-1] == self.input_dim, (
            f"Expected feature dimension {self.input_dim}, "
            f"got {x.shape[-1]}"
        )

        assert x.shape[1] == self.num_classes, (
            f"Expected {self.num_classes} disease representations, "
            f"got {x.shape[1]}"
        )

        # ---------------------------------------------------------------------
        # Layer normalization
        # ---------------------------------------------------------------------

        x = self.layer_norm(x)

        # ---------------------------------------------------------------------
        # Shared projection
        #
        # [B, 8, 768] → [B, 8, 512]
        # ---------------------------------------------------------------------

        x = self.shared_projection(x)

        # ---------------------------------------------------------------------
        # Independent disease heads
        # ---------------------------------------------------------------------

        logits = []

        for disease_idx, head in enumerate(self.disease_heads):

            disease_feature = x[:, disease_idx, :]

            disease_logit = head(
                disease_feature
            )

            logits.append(
                disease_logit
            )

        # [B, 8, 1] → [B, 8]

        logits = torch.cat(
            logits,
            dim=1
        )

        return logits


# =============================================================================
# 3. INSTANTIATE MODEL
# =============================================================================

model = RepresentationAwareMultiLabelHead(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)

print("\n" + "-" * 80)
print("MODEL CREATED")
print("-" * 80)

print(model)


# =============================================================================
# 4. PARAMETER COUNT
# =============================================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "-" * 80)
print("PARAMETER COUNT")
print("-" * 80)

print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)


# =============================================================================
# 5. SHAPE TEST
# =============================================================================

print("\n" + "-" * 80)
print("FORWARD-PASS SHAPE TEST")
print("-" * 80)

model.eval()

with torch.no_grad():

    test_input = torch.randn(
        4,
        NUM_CLASSES,
        INPUT_DIM
    )

    test_output = model(
        test_input
    )

print(
    f"Input shape  : {tuple(test_input.shape)}"
)

print(
    f"Output shape : {tuple(test_output.shape)}"
)

assert test_output.shape == (
    4,
    NUM_CLASSES
)

print("\n✓ Forward pass successful.")
print("✓ Output contains 8 independent logits.")


# =============================================================================
# 6. VERIFY DISEASE-SLOT INDEPENDENCE
# =============================================================================

print("\n" + "-" * 80)
print("DISEASE-SLOT INDEPENDENCE TEST")
print("-" * 80)

# Change only disease slot 0.
# Other disease representations remain unchanged.

x_a = torch.randn(
    2,
    NUM_CLASSES,
    INPUT_DIM
)

x_b = x_a.clone()

x_b[:, 0, :] += 5.0

model.eval()

with torch.no_grad():

    logits_a = model(x_a)
    logits_b = model(x_b)

difference = (
    logits_a - logits_b
).abs()

print(
    "Mean absolute logit difference by disease:"
)

for i, disease in enumerate(DISEASE_NAMES):

    print(
        f"  {disease}: "
        f"{difference[:, i].mean().item():.6f}"
    )


# =============================================================================
# 7. LOGIT OUTPUT VERIFICATION
# =============================================================================

assert torch.isfinite(
    test_output
).all()

assert test_output.ndim == 2
assert test_output.shape[1] == NUM_CLASSES

print("\n✓ Logits are finite.")
print("✓ Output dimensionality = 8.")
print("✓ No softmax applied.")
print("✓ Classes remain independently represented.")


# =============================================================================
# 8. SIGMOID INTERPRETATION CHECK
# =============================================================================

with torch.no_grad():

    probabilities = torch.sigmoid(
        test_output
    )

assert torch.all(
    probabilities >= 0
)

assert torch.all(
    probabilities <= 1
)

print("\n" + "-" * 80)
print("MULTI-LABEL PROBABILITY CHECK")
print("-" * 80)

print(
    f"Probability range: "
    f"{probabilities.min().item():.6f} "
    f"→ "
    f"{probabilities.max().item():.6f}"
)

print("\n✓ Sigmoid produces independent [0,1] probabilities.")




CELL 3 — REPRESENTATION-AWARE MULTI-LABEL CLASSIFICATION HEAD

--------------------------------------------------------------------------------
LOCKED INPUT CONFIGURATION
--------------------------------------------------------------------------------
Number of disease representations : 8
Representation dimension          : 768
Hidden dimension                  : 512
Dropout                           : 0.3

--------------------------------------------------------------------------------
MODEL CREATED
--------------------------------------------------------------------------------
RepresentationAwareMultiLabelHead(
  (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (shared_projection): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (disease_heads): ModuleList(
    (0-7): 8 x Linear(in_features=512, out_features=1, bias=True)
  )
)

------------------------------------------------

LOCKED UPSTREAM

Experiment E / Seed 42
Adaptive bilateral fusion
8 disease-specific representations
768 dimensions per representation
Upstream parameters FROZEN


CLASSIFICATION HEAD:

      Input                    : [B, 8, 768]
      Per-disease LayerNorm    : 768
      Shared projection        : 768 → 512
      Activation               : ReLU
      Dropout                  : 0.30

Disease-specific heads:

    N → 512 → 1
    D → 512 → 1
    G → 512 → 1
    C → 512 → 1
    A → 512 → 1
    H → 512 → 1
    M → 512 → 1
    O → 512 → 1

Output:

     [B, 8] logits
Probability              : Independent sigmoid

Loss                     : Asymmetric Loss (next cell)


DESIGN PRINCIPLE:

**The classifier preserves the disease-specific structure produced by
Experiment E instead of flattening all 8 × 768 features into one vector.
The shared projection controls parameter growth while the independent
heads allow each disease to learn its own decision boundary.**


CELL 4 — WEIGHTED RANDOM SAMPLING


OBJECTIVE

Address patient-level class imbalance during classifier training.

METHOD

WeightedRandomSampler is used for the training set.

Sampling weights are calculated from the patient's complete multi-label
target rather than treating each disease independently.

Rare disease labels receive greater sampling importance, while patients
containing multiple disease labels can contribute to multiple class
requirements.

Patients with no positive disease label retain a baseline sampling weight.

TRAINING:

    Batch size                 : 64
    Sampling                  : WeightedRandomSampler
    Replacement               : Yes
    Samples per epoch         : 2141

VALIDATION:

    Sampling                  : None
    Shuffle                   : No
    Distribution              : Natural validation distribution

This prevents the validation metrics from being artificially affected by
class-balancing during evaluation.

BALANCING STRATEGY

Patient-level balancing:
    WeightedRandomSampler

Example-level loss handling:
    Asymmetric Loss

These address different parts of the imbalance problem and will be evaluated
together during classifier training.

IMPORTANT

**No data is duplicated physically.
No validation patients are resampled.
No labels are modified.
No upstream Experiment E representation is modified.**



In [6]:
# =============================================================================
# CELL 4 — WEIGHTED RANDOM SAMPLING
# =============================================================================

from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

BATCH_SIZE = 64

y = Y_train.float()

class_counts = y.sum(dim=0)
class_weights = y.shape[0] / (class_counts * NUM_CLASSES)

sample_weights = (y * class_weights).sum(dim=1)

# Give all-zero-label patients a normal baseline weight
zero_label = y.sum(dim=1) == 0
sample_weights[zero_label] = 1.0

sample_weights = sample_weights / sample_weights.mean()

sampler = WeightedRandomSampler(
    weights=sample_weights.double(),
    num_samples=len(sample_weights),
    replacement=True
)

train_dataset = TensorDataset(
    X_train.float(),
    Y_train.float()
)

val_dataset = TensorDataset(
    X_val.float(),
    Y_val.float()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Weighted sampling configured.")
print(f"Batch size: {BATCH_SIZE}")
print(f"Train samples per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Sample weight range: {sample_weights.min():.3f} - {sample_weights.max():.3f}")

assert len(train_dataset) == 2141
assert len(val_dataset) == 438
assert isinstance(train_loader.sampler, WeightedRandomSampler)
assert not isinstance(val_loader.sampler, WeightedRandomSampler)

print("✓ Train: WeightedRandomSampler")
print("✓ Validation: natural distribution")
print("✓ No validation resampling")

Weighted sampling configured.
Batch size: 64
Train samples per epoch: 34
Validation batches: 7
Sample weight range: 0.367 - 6.431
✓ Train: WeightedRandomSampler
✓ Validation: natural distribution
✓ No validation resampling


CELL 5 — INITIAL MULTI-LABEL CLASSIFIER TRAINING
OBJECTIVE

Train the first classifier using the locked Experiment E representations.

TRAINING:

    Optimizer: AdamW
    Learning rate: 0.001
    Weight decay: 0.0001
    Batch size: 64
    Maximum epochs: 40
    Early stopping patience: 8

IMBALANCE HANDLING:

    WeightedRandomSampler for training
    Asymmetric Loss for optimization
    Validation remains naturally distributed

MODEL SELECTION:

    Primary metric: Validation Macro-F1

    Also record:

    Micro-F1
    Validation loss
    Exact Match
    Per-disease F1
    THRESHOLD

    Initial threshold: 0.50

Validation probabilities will be retained for later threshold optimization without retraining.

**IMPORTANT**

*Experiment E, Seed 42 and all fused representations remain frozen. No upstream modification is performed.*

In [7]:
# =============================================================================
# CELL 5 — INITIAL MULTI-LABEL CLASSIFIER TRAINING
# =============================================================================

import copy
import numpy as np
import torch.nn as nn
from sklearn.metrics import f1_score


class AsymmetricLoss(nn.Module):

    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):

        xs_pos = torch.sigmoid(logits)
        xs_neg = 1.0 - xs_pos

        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        xs_pos = xs_pos.clamp(self.eps, 1.0 - self.eps)
        xs_neg = xs_neg.clamp(self.eps, 1.0 - self.eps)

        loss = (
            targets * torch.log(xs_pos) +
            (1.0 - targets) * torch.log(xs_neg)
        )

        asymmetric_weight = (
            targets * (1.0 - xs_pos.detach()).pow(self.gamma_pos) +
            (1.0 - targets) * (1.0 - xs_neg.detach()).pow(self.gamma_neg)
        )

        return -(loss * asymmetric_weight).mean()


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 8

model = RepresentationAwareMultiLabelHead(
    input_dim=768,
    hidden_dim=512,
    num_classes=8,
    dropout=0.30
).to(DEVICE)

criterion = AsymmetricLoss(
    gamma_neg=4.0,
    gamma_pos=1.0,
    clip=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

best_macro_f1 = -1.0
best_state = None
best_epoch = 0
history = []
patience_counter = 0

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()

    val_loss = 0.0
    all_probs = []
    all_targets = []

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)

            all_probs.append(
                torch.sigmoid(logits).cpu().numpy()
            )

            all_targets.append(
                yb.cpu().numpy()
            )

    val_loss /= len(val_loader.dataset)

    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)

    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(
        targets, preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        targets, preds,
        average="macro",
        zero_division=0
    )

    exact_match = np.mean(
        np.all(preds == targets, axis=1)
    )

    per_class_f1 = f1_score(
        targets, preds,
        average=None,
        zero_division=0
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "exact_match": exact_match,
        "per_class_f1": per_class_f1.copy()
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"Micro {micro_f1:.4f} | "
        f"Macro {macro_f1:.4f} | "
        f"EM {exact_match:.4f}"
    )

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        best_probs = probs.copy()
        best_targets = targets.copy()
        patience_counter = 0

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break


model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Macro-F1: {best_macro_f1:.4f}")

Epoch 01 | Train 0.0453 | Val 0.0572 | Micro 0.4784 | Macro 0.5056 | EM 0.0411
Epoch 02 | Train 0.0308 | Val 0.0606 | Micro 0.4878 | Macro 0.5205 | EM 0.0639
Epoch 03 | Train 0.0264 | Val 0.0626 | Micro 0.5030 | Macro 0.5359 | EM 0.0571
Epoch 04 | Train 0.0234 | Val 0.0642 | Micro 0.5146 | Macro 0.5318 | EM 0.0959
Epoch 05 | Train 0.0211 | Val 0.0674 | Micro 0.5266 | Macro 0.5520 | EM 0.1005
Epoch 06 | Train 0.0216 | Val 0.0755 | Micro 0.5216 | Macro 0.5528 | EM 0.0936
Epoch 07 | Train 0.0186 | Val 0.0657 | Micro 0.5323 | Macro 0.5533 | EM 0.1256
Epoch 08 | Train 0.0181 | Val 0.0718 | Micro 0.5307 | Macro 0.5554 | EM 0.1164
Epoch 09 | Train 0.0171 | Val 0.0743 | Micro 0.5271 | Macro 0.5438 | EM 0.1164
Epoch 10 | Train 0.0162 | Val 0.0756 | Micro 0.5485 | Macro 0.5664 | EM 0.1324
Epoch 11 | Train 0.0160 | Val 0.0765 | Micro 0.5437 | Macro 0.5601 | EM 0.1347
Epoch 12 | Train 0.0146 | Val 0.0802 | Micro 0.5380 | Macro 0.5549 | EM 0.1279
Epoch 13 | Train 0.0147 | Val 0.0838 | Micro 0.5424 

###EXPERIMENT 2

CELL 6 — EXPERIMENT F CLASSIFIER ARCHITECTURE


CONFIGURATION

    Experiment              : 2
    Upstream representation : Experiment E / Seed 42
    Upstream fusion         : FROZEN

ARCHITECTURAL CHANGES

    1. LayerNorm on the 768-D disease-specific representations.

    2. Double-dropout bottleneck:
          LayerNorm
              ↓
          Dropout(0.3)
              ↓
          FC(768 → 512)
              ↓
          ReLU
              ↓
          Dropout(0.3)

    3. Global disease-context pathway:
          Mean over 8 disease representations
              ↓
          FC(768 → 512)
              ↓
          Context added to every disease-specific 512-D representation.

    4. Eight independent classification outputs.


OUTPUT

Input  : [B, 8, 768]

Output : [B, 8]


Each output corresponds independently to:
    *N, D, G, C, A, H, M, O*


TRAINING CHANGES FOR EXPERIMENT 2

    Sampling              : WeightedRandomSampler
    ASL gamma_neg         : 2.0
    ASL gamma_pos         : 1.0
    ASL clipping          : 0.05
    AdamW weight decay    : 1e-2

The locked Experiment E fused representations are not modified.



In [8]:
# =============================================================================
# CELL 6 — EXPERIMENT 2 CLASSIFIER ARCHITECTURE
# =============================================================================

import torch
import torch.nn as nn


class RepresentationAwareMultiLabelHeadF(nn.Module):

    def __init__(self, input_dim=768, hidden_dim=512, num_classes=8, dropout=0.30):
        super().__init__()

        self.norm = nn.LayerNorm(input_dim)
        self.input_dropout = nn.Dropout(dropout)

        self.shared_projection = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.hidden_dropout = nn.Dropout(dropout)

        self.context_projection = nn.Linear(input_dim, hidden_dim)

        self.disease_heads = nn.ModuleList([
            nn.Linear(hidden_dim, 1)
            for _ in range(num_classes)
        ])

    def forward(self, x):

        x = self.norm(x)
        x = self.input_dropout(x)

        disease_features = self.shared_projection(x)
        disease_features = self.activation(disease_features)
        disease_features = self.hidden_dropout(disease_features)

        global_context = x.mean(dim=1)
        global_context = self.context_projection(global_context)
        global_context = global_context.unsqueeze(1)

        disease_features = disease_features + global_context

        logits = torch.cat(
            [head(disease_features[:, i, :])
             for i, head in enumerate(self.disease_heads)],
            dim=1
        )

        return logits


model_2 = RepresentationAwareMultiLabelHeadF(
    input_dim=768,
    hidden_dim=512,
    num_classes=8,
    dropout=0.30
).to(DEVICE)

param_count = sum(
    p.numel() for p in model_2.parameters()
    if p.requires_grad
)

test_logits = model_2(X_train[:4].float().to(DEVICE))

print(f"Experiment 2 parameters: {param_count:,}")
print(f"Test output shape: {tuple(test_logits.shape)}")

assert test_logits.shape == (4, 8)
assert param_count > 0

print("✓ Double-dropout bottleneck verified")
print("✓ Global disease-context pathway verified")
print("✓ Eight independent outputs verified")
print("✓ Experiment 2 architecture ready")

Experiment 2 parameters: 793,096
Test output shape: (4, 8)
✓ Double-dropout bottleneck verified
✓ Global disease-context pathway verified
✓ Eight independent outputs verified
✓ Experiment 2 architecture ready


CELL 7 — EXPERIMENT 2 TRAINING


OBJECTIVE

**Train the revised multi-label classification head using the locked
Experiment E representations.**

EXPERIMENT 2 CHANGES

    Architecture:
        • Double-dropout bottleneck
        • Global cross-disease context injection

    Optimization:
        • AdamW weight decay = 1e-2

    Loss:
        • Asymmetric Loss
        • gamma_neg = 2.0
        • gamma_pos = 1.0
        • clip = 0.05

    Sampling:
        • WeightedRandomSampler
        • Training only

CONTROL

    Experiment 1 remains the baseline reference.

    Experiment 2 does not modify:
        • Experiment E
        • fused representations
        • cross-eye features
        • patient cohorts
        • disease labels

MODEL SELECTION

    Primary metric : Validation Macro-F1

    Also recorded:
        • Validation loss
        • Micro-F1
        • Exact Match
        • Per-disease F1

    The best Macro-F1 checkpoint is retained.

THRESHOLD

Initial threshold = 0.50

Validation probabilities are retained for later threshold optimization.



In [9]:
# =============================================================================
# CELL 7 — EXPERIMENT 2 TRAINING
# =============================================================================

import copy
import numpy as np
from sklearn.metrics import f1_score

EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-2
PATIENCE = 8

criterion = AsymmetricLoss(
    gamma_neg=2.0,
    gamma_pos=1.0,
    clip=0.05
)

model = model_2.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

best_macro_f1 = -1.0
best_state = None
best_epoch = 0
best_probs = None
best_targets = None
history_exp2 = []
patience_counter = 0

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    all_probs = []
    all_targets = []

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)

            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(yb.cpu().numpy())

    val_loss /= len(val_loader.dataset)

    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)

    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(
        targets, preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        targets, preds,
        average="macro",
        zero_division=0
    )

    exact_match = np.mean(
        np.all(preds == targets, axis=1)
    )

    per_class_f1 = f1_score(
        targets, preds,
        average=None,
        zero_division=0
    )

    history_exp2.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "exact_match": exact_match,
        "per_class_f1": per_class_f1.copy()
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"Micro {micro_f1:.4f} | "
        f"Macro {macro_f1:.4f} | "
        f"EM {exact_match:.4f}"
    )

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        best_probs = probs.copy()
        best_targets = targets.copy()
        patience_counter = 0

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Macro-F1: {best_macro_f1:.4f}")


Epoch 01 | Train 0.0825 | Val 0.1137 | Micro 0.4985 | Macro 0.5493 | EM 0.1826
Epoch 02 | Train 0.0542 | Val 0.1086 | Micro 0.5307 | Macro 0.5629 | EM 0.2123
Epoch 03 | Train 0.0474 | Val 0.1011 | Micro 0.5331 | Macro 0.5699 | EM 0.2352
Epoch 04 | Train 0.0413 | Val 0.1065 | Micro 0.5467 | Macro 0.5615 | EM 0.2009
Epoch 05 | Train 0.0403 | Val 0.1117 | Micro 0.5060 | Macro 0.5398 | EM 0.2215
Epoch 06 | Train 0.0371 | Val 0.1083 | Micro 0.5466 | Macro 0.5684 | EM 0.2603
Epoch 07 | Train 0.0365 | Val 0.1168 | Micro 0.5186 | Macro 0.5368 | EM 0.2511
Epoch 08 | Train 0.0359 | Val 0.1049 | Micro 0.5637 | Macro 0.5636 | EM 0.2740
Epoch 09 | Train 0.0325 | Val 0.1133 | Micro 0.5415 | Macro 0.5610 | EM 0.2557
Epoch 10 | Train 0.0331 | Val 0.1184 | Micro 0.5510 | Macro 0.5734 | EM 0.1849
Epoch 11 | Train 0.0306 | Val 0.1194 | Micro 0.5464 | Macro 0.5661 | EM 0.2694
Epoch 12 | Train 0.0319 | Val 0.1235 | Micro 0.5538 | Macro 0.5612 | EM 0.2374
Epoch 13 | Train 0.0315 | Val 0.1192 | Micro 0.5620 

###EXPERIMENT 3



CELL 8 — EXPERIMENT 3 TRAINING CONFIGURATION

UPSTREAM

Experiment E / Seed 42 fused representations remain frozen.

ARCHITECTURE

    Experiment 2 architecture retained:
        • LayerNorm
        • Double Dropout(0.3)
        • FC 768 → 512
        • ReLU
        • Global cross-disease context
        • Eight independent disease heads

LOSS

    Asymmetric Loss:
        gamma_neg = 1.5
        gamma_pos = 0.5
        clip      = 0.05

    Label smoothing:
        0 → 0.05
        1 → 0.95

OPTIMIZATION

    Optimizer : AdamW
    Weight decay : 1e-2

Sampling:

    WeightedRandomSampler
    Training only

SWA

**Stochastic Weight Averaging will be applied during the later training phase.**

EVALUATION

    Primary metric:
        Validation Macro-F1

    Also recorded:
        Micro-F1
        Exact Match
        Per-disease F1
        Validation loss

THRESHOLD CALIBRATION

    Initial evaluation:
        threshold = 0.50

    Post-training calibration:
        • Search each disease independently.
        • Candidate thresholds: 0.05–0.95
        • Step: 0.01
        • Objective: maximize per-disease F1.

The optimized 8-threshold vector will then be used to
recalculate Macro-F1, Micro-F1 and Exact Match.

Original binary labels remain unchanged.



In [10]:
# =============================================================================
# CELL 8 — EXPERIMENT 3 CONFIGURATION
# =============================================================================

E3_GAMMA_NEG = 1.5
E3_GAMMA_POS = 0.5
E3_CLIP = 0.05
E3_LABEL_SMOOTH = 0.05

E3_LR = 1e-3
E3_WEIGHT_DECAY = 1e-2
E3_EPOCHS = 40
E3_PATIENCE = 8

E3_SWA_START = 20

DISEASE_NAMES = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

# criterion_exp3 = AsymmetricLoss(
#     gamma_neg=E3_GAMMA_NEG,
#     gamma_pos=E3_GAMMA_POS,
#     clip=E3_CLIP,
#     label_smoothing=E3_LABEL_SMOOTH
# )

model_exp3 = RepresentationAwareMultiLabelHeadF(
    input_dim=768,
    hidden_dim=512,
    num_classes=8,
    dropout=0.30
).to(DEVICE)

optimizer_exp3 = torch.optim.AdamW(
    model_exp3.parameters(),
    lr=E3_LR,
    weight_decay=E3_WEIGHT_DECAY
)

swa_model_exp3 = torch.optim.swa_utils.AveragedModel(model_exp3)
swa_scheduler_exp3 = torch.optim.swa_utils.SWALR(
    optimizer_exp3,
    swa_lr=E3_LR * 0.1
)

print("Experiment 3 configuration ready.")
print(f"ASL: gamma_neg={E3_GAMMA_NEG}, gamma_pos={E3_GAMMA_POS}")
print(f"Label smoothing: {E3_LABEL_SMOOTH}")
print(f"Weight decay: {E3_WEIGHT_DECAY}")
print(f"SWA start epoch: {E3_SWA_START}")

Experiment 3 configuration ready.
ASL: gamma_neg=1.5, gamma_pos=0.5
Label smoothing: 0.05
Weight decay: 0.01
SWA start epoch: 20


CELL 8A — EXPERIMENT 3 ASYMMETRIC LOSS


Label smoothing is applied only inside Experiment 3 loss computation.

Original targets remain unchanged:

    0 → 0.05
    1 → 0.95

ASL:

    gamma_neg = 1.5
    gamma_pos = 0.5
    clip      = 0.05

No training data or validation labels are modified.



In [11]:
class AsymmetricLossExp3(nn.Module):

    def __init__(
        self,
        gamma_neg=1.5,
        gamma_pos=0.5,
        clip=0.05,
        label_smoothing=0.05
    ):
        super().__init__()

        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):

        targets = targets.float()

        if self.label_smoothing > 0:
            targets = (
                targets * (1.0 - 2.0 * self.label_smoothing)
                + self.label_smoothing
            )

        xs_pos = torch.sigmoid(logits)
        xs_neg = 1.0 - xs_pos

        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        loss_pos = targets * torch.log(xs_pos.clamp(min=1e-8))
        loss_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=1e-8))

        loss = loss_pos + loss_neg

        if self.gamma_neg > 0 or self.gamma_pos > 0:

            pt = xs_pos * targets + xs_neg * (1.0 - targets)
            one_sided_gamma = (
                self.gamma_pos * targets
                + self.gamma_neg * (1.0 - targets)
            )

            one_sided_w = torch.pow(
                1.0 - pt,
                one_sided_gamma
            )

            loss *= one_sided_w

        return -loss.mean()


criterion_exp3 = AsymmetricLossExp3(
    gamma_neg=1.5,
    gamma_pos=0.5,
    clip=0.05,
    label_smoothing=0.05
)

test_loss = criterion_exp3(
    torch.randn(4, 8, device=DEVICE),
    torch.randint(0, 2, (4, 8), device=DEVICE)
)

assert torch.isfinite(test_loss)

print("✓ Experiment 3 ASL ready")
print(f"✓ gamma_neg = {criterion_exp3.gamma_neg}")
print(f"✓ gamma_pos = {criterion_exp3.gamma_pos}")
print(f"✓ label smoothing = {criterion_exp3.label_smoothing}")

✓ Experiment 3 ASL ready
✓ gamma_neg = 1.5
✓ gamma_pos = 0.5
✓ label smoothing = 0.05



CELL 9 — EXPERIMENT 3 TRAINING WITH SWA


Experiment 3 is trained for the full 40-epoch budget.

SWA:

    Start epoch : 25
    Averaging   : epochs 25–40

Early stopping is disabled so that SWA receives a complete averaging window.

Training:

    • WeightedRandomSampler
    • AdamW, weight decay = 0.01
    • ASL: gamma_neg = 1.5, gamma_pos = 0.5
    • Label smoothing = 0.05

Validation:

    • Natural distribution
    • Threshold = 0.50 for training-time metrics
    • No test data used

Both the best conventional checkpoint and the final SWA model are evaluated.
The stronger model will be used for subsequent per-disease threshold
optimization.




In [12]:
# =============================================================================
# CELL 9 — EXPERIMENT 3 TRAINING + SWA
# =============================================================================


E3_MAX_EPOCHS = 40
E3_SWA_START = 25

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
history_exp3 = []

swa_model_exp3 = torch.optim.swa_utils.AveragedModel(model_exp3).to(DEVICE)

for epoch in range(1, E3_MAX_EPOCHS + 1):

    model_exp3.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer_exp3.zero_grad()
        logits = model_exp3(xb)
        loss = criterion_exp3(logits, yb)
        loss.backward()
        optimizer_exp3.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    if epoch >= E3_SWA_START:
        swa_model_exp3.update_parameters(model_exp3)

    model_exp3.eval()
    val_loss = 0.0
    probs_list = []
    targets_list = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            logits = model_exp3(xb)
            loss = criterion_exp3(logits, yb)

            val_loss += loss.item() * xb.size(0)
            probs_list.append(torch.sigmoid(logits).cpu().numpy())
            targets_list.append(yb.cpu().numpy())

    val_loss /= len(val_loader.dataset)

    probs = np.concatenate(probs_list)
    targets = np.concatenate(targets_list)
    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(
        targets, preds, average="micro", zero_division=0
    )
    macro_f1 = f1_score(
        targets, preds, average="macro", zero_division=0
    )
    exact_match = np.mean(np.all(preds == targets, axis=1))

    history_exp3.append([
        epoch, train_loss, val_loss,
        micro_f1, macro_f1, exact_match
    ])

    print(
        f"Epoch {epoch:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"Micro {micro_f1:.4f} | "
        f"Macro {macro_f1:.4f} | "
        f"EM {exact_match:.4f}"
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model_exp3.state_dict())


torch.optim.swa_utils.update_bn(
    train_loader,
    swa_model_exp3,
    device=DEVICE
)

swa_model_exp3.eval()

swa_probs_list = []
swa_targets_list = []

with torch.no_grad():
    for xb, yb in val_loader:
        logits = swa_model_exp3(xb.to(DEVICE))
        swa_probs_list.append(torch.sigmoid(logits).cpu().numpy())
        swa_targets_list.append(yb.numpy())

swa_probs = np.concatenate(swa_probs_list)
swa_targets = np.concatenate(swa_targets_list)
swa_preds = (swa_probs >= 0.5).astype(int)

swa_micro_f1 = f1_score(
    swa_targets, swa_preds, average="micro", zero_division=0
)
swa_macro_f1 = f1_score(
    swa_targets, swa_preds, average="macro", zero_division=0
)
swa_exact_match = np.mean(
    np.all(swa_preds == swa_targets, axis=1)
)

print(
    f"SWA | Micro {swa_micro_f1:.4f} | "
    f"Macro {swa_macro_f1:.4f} | "
    f"EM {swa_exact_match:.4f}"
)

print(f"Best conventional epoch: {best_epoch}")
print(f"Best conventional Macro-F1: {best_macro_f1:.4f}")

Epoch 01 | Train 0.1191 | Val 0.1598 | Micro 0.4540 | Macro 0.5154 | EM 0.2100
Epoch 02 | Train 0.0802 | Val 0.1571 | Micro 0.4720 | Macro 0.5260 | EM 0.2009
Epoch 03 | Train 0.0686 | Val 0.1468 | Micro 0.5118 | Macro 0.5424 | EM 0.2877
Epoch 04 | Train 0.0601 | Val 0.1536 | Micro 0.5214 | Macro 0.5465 | EM 0.2580
Epoch 05 | Train 0.0617 | Val 0.1432 | Micro 0.5149 | Macro 0.5373 | EM 0.3288
Epoch 06 | Train 0.0593 | Val 0.1515 | Micro 0.5297 | Macro 0.5509 | EM 0.2900
Epoch 07 | Train 0.0544 | Val 0.1428 | Micro 0.5537 | Macro 0.5730 | EM 0.3265
Epoch 08 | Train 0.0507 | Val 0.1551 | Micro 0.5414 | Macro 0.5598 | EM 0.2808
Epoch 09 | Train 0.0534 | Val 0.1548 | Micro 0.5209 | Macro 0.5555 | EM 0.2945
Epoch 10 | Train 0.0513 | Val 0.1491 | Micro 0.5321 | Macro 0.5609 | EM 0.3333
Epoch 11 | Train 0.0501 | Val 0.1571 | Micro 0.5558 | Macro 0.5857 | EM 0.2831
Epoch 12 | Train 0.0503 | Val 0.1584 | Micro 0.5590 | Macro 0.5731 | EM 0.3082
Epoch 13 | Train 0.0458 | Val 0.1649 | Micro 0.5372 

In [13]:
# =============================================================================
# CELL 10 — DIAGNOSTIC: DISEASE-SLOT REPRESENTATION SIMILARITY
# =============================================================================
# Tests whether the 8 disease-specific [768] slots per patient are actually
# distinct, or whether they're near-duplicates of the same two eye vectors
# reweighted by a near-uniform gate. No training involved.

import torch
import torch.nn.functional as F

print("=" * 80)
print("CELL 10 — DISEASE-SLOT SIMILARITY DIAGNOSTIC")
print("=" * 80)

def slot_similarity_report(X, split_name):
    normed = F.normalize(X, dim=-1)                       # [N, 8, 768]
    sim = torch.einsum('nid,njd->nij', normed, normed)    # [N, 8, 8]
    mask = ~torch.eye(NUM_CLASSES, dtype=torch.bool)
    off_diag = sim[:, mask]                                # [N, 56]

    print(f"\n{split_name} — pairwise cosine similarity across disease slots")
    print(f"  mean : {off_diag.mean().item():.4f}")
    print(f"  std  : {off_diag.std().item():.4f}")
    print(f"  min  : {off_diag.min().item():.4f}")
    print(f"  max  : {off_diag.max().item():.4f}")
    return off_diag

train_sim = slot_similarity_report(X_train.float(), "Train")
val_sim   = slot_similarity_report(X_val.float(),   "Val")

print("\nInterpretation:")
print("  mean > 0.95 -> slots are near-duplicates; disease heads are mostly")
print("    separating on their own learned bias, not on distinct content.")
print("  mean < 0.80 -> meaningful per-disease content exists; head-level")
print("    regularization is the right lever.")

assert torch.isfinite(train_sim).all() and torch.isfinite(val_sim).all()
print("\n✓ Similarity computed on locked Experiment E representations")
print("✓ No upstream modification")

CELL 10 — DISEASE-SLOT SIMILARITY DIAGNOSTIC

Train — pairwise cosine similarity across disease slots
  mean : 0.9985
  std  : 0.0027
  min  : 0.8915
  max  : 1.0000

Val — pairwise cosine similarity across disease slots
  mean : 0.9984
  std  : 0.0031
  min  : 0.9005
  max  : 1.0000

Interpretation:
  mean > 0.95 -> slots are near-duplicates; disease heads are mostly
    separating on their own learned bias, not on distinct content.
  mean < 0.80 -> meaningful per-disease content exists; head-level
    regularization is the right lever.

✓ Similarity computed on locked Experiment E representations
✓ No upstream modification


In [14]:
# =============================================================================
# CELL 11 — PER-DISEASE THRESHOLD OPTIMIZATION (NO RETRAINING)
# =============================================================================

import numpy as np
from sklearn.metrics import f1_score

def optimize_thresholds(probs, targets, disease_names, grid=None):
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.01)

    best_thresholds = np.zeros(len(disease_names))
    best_f1s = np.zeros(len(disease_names))

    for i, name in enumerate(disease_names):
        scores = [
            f1_score(targets[:, i], (probs[:, i] >= t).astype(int), zero_division=0)
            for t in grid
        ]
        best_idx = int(np.argmax(scores))
        best_thresholds[i] = grid[best_idx]
        best_f1s[i] = scores[best_idx]

    return best_thresholds, best_f1s


def apply_thresholds(probs, targets, thresholds, disease_names, label=""):
    preds = (probs >= thresholds[None, :]).astype(int)

    micro = f1_score(targets, preds, average="micro", zero_division=0)
    macro = f1_score(targets, preds, average="macro", zero_division=0)
    exact = np.mean(np.all(preds == targets, axis=1))
    per_class = f1_score(targets, preds, average=None, zero_division=0)

    print(f"\n--- {label} (calibrated thresholds) ---")
    print(f"Micro-F1: {micro:.4f} | Macro-F1: {macro:.4f} | Exact Match: {exact:.4f}")
    for name, t, f1 in zip(disease_names, thresholds, per_class):
        print(f"  {name}: threshold={t:.2f}  F1={f1:.4f}")

    return preds, micro, macro, exact


print("=" * 80)
print("CELL 11 — PER-DISEASE THRESHOLD OPTIMIZATION")
print("=" * 80)

# Run separately for each experiment's saved probs/targets — rename these
# per experiment going forward (see the variable-collision note above).
thresholds, _ = optimize_thresholds(best_probs, best_targets, DISEASE_NAMES)

base_preds = (best_probs >= 0.5).astype(int)
print(f"\n--- Baseline (threshold=0.50) ---")
print(f"Macro-F1: {f1_score(best_targets, base_preds, average='macro', zero_division=0):.4f}")

apply_thresholds(best_probs, best_targets, thresholds, DISEASE_NAMES, label="Current model")

CELL 11 — PER-DISEASE THRESHOLD OPTIMIZATION

--- Baseline (threshold=0.50) ---
Macro-F1: 0.5770

--- Current model (calibrated thresholds) ---
Micro-F1: 0.5845 | Macro-F1: 0.6072 | Exact Match: 0.2420
  N: threshold=0.43  F1=0.5782
  D: threshold=0.49  F1=0.6287
  G: threshold=0.49  F1=0.6769
  C: threshold=0.60  F1=0.8085
  A: threshold=0.94  F1=0.5000
  H: threshold=0.71  F1=0.3448
  M: threshold=0.52  F1=0.8444
  O: threshold=0.48  F1=0.4762


(array([[1, 1, 0, ..., 0, 0, 1],
        [0, 1, 0, ..., 0, 1, 0],
        [0, 0, 0, ..., 0, 1, 0],
        ...,
        [1, 1, 0, ..., 0, 0, 1],
        [0, 1, 0, ..., 0, 0, 1],
        [0, 1, 0, ..., 0, 0, 0]]),
 0.5844748858447488,
 0.6072323037774903,
 np.float64(0.2420091324200913))

In [15]:
# =============================================================================
# CELL 12 — PER-DISEASE F1 AT BEST EPOCH
# =============================================================================

def print_per_disease(history, label):
    best = max(history, key=lambda h: h["macro_f1"])
    print(f"\n{label} — best epoch {best['epoch']} (Macro-F1 {best['macro_f1']:.4f})")
    for name, f1 in zip(DISEASE_NAMES, best["per_class_f1"]):
        print(f"  {name}: {f1:.4f}")

print_per_disease(history, "Experiment 1")
print_per_disease(history_exp2, "Experiment 2")


Experiment 1 — best epoch 19 (Macro-F1 0.5837)
  N: 0.5700
  D: 0.6087
  G: 0.6349
  C: 0.7843
  A: 0.4000
  H: 0.3784
  M: 0.8511
  O: 0.4424

Experiment 2 — best epoch 13 (Macro-F1 0.5770)
  N: 0.5586
  D: 0.6181
  G: 0.6769
  C: 0.7917
  A: 0.4091
  H: 0.2857
  M: 0.8261
  O: 0.4497


In [16]:
# =============================================================================
# CELL 13 — BOOTSTRAP CONFIDENCE INTERVAL ON MACRO-F1
# =============================================================================
# With 438 validation patients, 0.01-0.02 Macro-F1 differences between runs
# are frequently noise. This resamples patients (not retraining) to get a
# 95% CI so you know whether a change is real before running another
# experiment chasing it.

import numpy as np
from sklearn.metrics import f1_score

def bootstrap_macro_f1_ci(probs, targets, threshold=0.5, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(targets)
    preds = (probs >= threshold).astype(int)
    scores = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores[b] = f1_score(targets[idx], preds[idx], average="macro", zero_division=0)

    point = f1_score(targets, preds, average="macro", zero_division=0)
    lo, hi = np.percentile(scores, [2.5, 97.5])
    print(f"Macro-F1: {point:.4f}   95% CI: [{lo:.4f}, {hi:.4f}]")
    return point, lo, hi

bootstrap_macro_f1_ci(best_probs, best_targets)

Macro-F1: 0.5770   95% CI: [0.5291, 0.6212]


(0.576974416908343,
 np.float64(0.5291173039352612),
 np.float64(0.6212362257079634))

EXPERIMENT 4

In [17]:
# =============================================================================
# ###EXPERIMENT 4
# CELL 14 — EXPERIMENT 4 ARCHITECTURE (LEAN HEAD, NO CROSS-DISEASE CONTEXT)
# =============================================================================

import torch
import torch.nn as nn

USE_CONTEXT = False  # off by default — see collapse diagnostic reasoning above

class RepresentationAwareMultiLabelHeadV4(nn.Module):

    def __init__(self, input_dim=768, hidden_dim=128, num_classes=8, dropout=0.40, use_context=False):
        super().__init__()

        self.use_context = use_context

        self.norm = nn.LayerNorm(input_dim)
        self.input_dropout = nn.Dropout(dropout)

        self.shared_projection = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.hidden_dropout = nn.Dropout(dropout)

        if use_context:
            self.context_projection = nn.Linear(input_dim, hidden_dim)

        self.disease_heads = nn.ModuleList([
            nn.Linear(hidden_dim, 1)
            for _ in range(num_classes)
        ])

    def forward(self, x):
        x = self.norm(x)
        x = self.input_dropout(x)

        disease_features = self.shared_projection(x)
        disease_features = self.activation(disease_features)
        disease_features = self.hidden_dropout(disease_features)

        if self.use_context:
            global_context = x.mean(dim=1)
            global_context = self.context_projection(global_context)
            disease_features = disease_features + global_context.unsqueeze(1)

        logits = torch.cat(
            [head(disease_features[:, i, :]) for i, head in enumerate(self.disease_heads)],
            dim=1
        )
        return logits


model_exp4 = RepresentationAwareMultiLabelHeadV4(
    input_dim=768,
    hidden_dim=128,
    num_classes=8,
    dropout=0.40,
    use_context=USE_CONTEXT
).to(DEVICE)

param_count = sum(p.numel() for p in model_exp4.parameters() if p.requires_grad)
test_logits = model_exp4(X_train[:4].float().to(DEVICE))

print(f"Experiment 4 parameters: {param_count:,}")
print(f"Test output shape: {tuple(test_logits.shape)}")
assert test_logits.shape == (4, 8)
print(f"✓ Cross-disease context path: {'ENABLED' if USE_CONTEXT else 'DISABLED'}")
print("✓ Experiment 4 architecture ready")

Experiment 4 parameters: 101,000
Test output shape: (4, 8)
✓ Cross-disease context path: DISABLED
✓ Experiment 4 architecture ready


In [18]:
# =============================================================================
# CELL 15 — EXPERIMENT 4 TRAINING (CHECKPOINT AVERAGING, SMOOTHED SELECTION)
# =============================================================================

import copy
import numpy as np
from collections import deque
from sklearn.metrics import f1_score

E4_EPOCHS = 40
E4_LR = 1e-3
E4_WEIGHT_DECAY = 1e-2
E4_PATIENCE = 8
E4_SMOOTH_WINDOW = 3   # moving-average window for model-selection metric
E4_TOPK = 3            # number of best checkpoints to average at the end

criterion_exp4 = AsymmetricLoss(gamma_neg=2.0, gamma_pos=1.0, clip=0.05)

optimizer_exp4 = torch.optim.AdamW(
    model_exp4.parameters(),
    lr=E4_LR,
    weight_decay=E4_WEIGHT_DECAY
)

exp4_history = []
recent_macro = deque(maxlen=E4_SMOOTH_WINDOW)
top_checkpoints = []   # list of (smoothed_macro_f1, state_dict)
best_smoothed_f1 = -1.0
patience_counter = 0

for epoch in range(1, E4_EPOCHS + 1):

    model_exp4.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer_exp4.zero_grad()
        logits = model_exp4(xb)
        loss = criterion_exp4(logits, yb)
        loss.backward()
        optimizer_exp4.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model_exp4.eval()
    val_loss = 0.0
    probs_list, targets_list = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model_exp4(xb)
            loss = criterion_exp4(logits, yb)

            val_loss += loss.item() * xb.size(0)
            probs_list.append(torch.sigmoid(logits).cpu().numpy())
            targets_list.append(yb.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    probs = np.concatenate(probs_list)
    targets = np.concatenate(targets_list)
    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(targets, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    exact_match = np.mean(np.all(preds == targets, axis=1))
    per_class_f1 = f1_score(targets, preds, average=None, zero_division=0)

    recent_macro.append(macro_f1)
    smoothed_macro_f1 = float(np.mean(recent_macro))

    exp4_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "smoothed_macro_f1": smoothed_macro_f1,
        "exact_match": exact_match,
        "per_class_f1": per_class_f1.copy()
    })

    print(
        f"Epoch {epoch:02d} | Train {train_loss:.4f} | Val {val_loss:.4f} | "
        f"Micro {micro_f1:.4f} | Macro {macro_f1:.4f} (smoothed {smoothed_macro_f1:.4f}) | "
        f"EM {exact_match:.4f}"
    )

    # Maintain top-K checkpoints by smoothed macro-F1 for end-of-run averaging
    top_checkpoints.append((smoothed_macro_f1, copy.deepcopy(model_exp4.state_dict())))
    top_checkpoints = sorted(top_checkpoints, key=lambda t: t[0], reverse=True)[:E4_TOPK]

    if smoothed_macro_f1 > best_smoothed_f1:
        best_smoothed_f1 = smoothed_macro_f1
        exp4_best_epoch = epoch
        exp4_best_state = copy.deepcopy(model_exp4.state_dict())
        exp4_best_probs = probs.copy()
        exp4_best_targets = targets.copy()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= E4_PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

model_exp4.load_state_dict(exp4_best_state)
print(f"\nBest epoch (smoothed selection): {exp4_best_epoch}")
print(f"Best smoothed Macro-F1: {best_smoothed_f1:.4f}")

# --- Checkpoint averaging (replaces the broken SWA path) ---
# Works regardless of when early stopping fires, unlike torch's schedule-based SWA.
avg_state = copy.deepcopy(top_checkpoints[0][1])
for key in avg_state:
    avg_state[key] = torch.stack([ckpt[1][key].float() for ckpt in top_checkpoints]).mean(dim=0)

model_exp4_avg = RepresentationAwareMultiLabelHeadV4(
    input_dim=768, hidden_dim=128, num_classes=8, dropout=0.40, use_context=USE_CONTEXT
).to(DEVICE)
model_exp4_avg.load_state_dict(avg_state)
model_exp4_avg.eval()

avg_probs_list, avg_targets_list = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        logits = model_exp4_avg(xb)
        avg_probs_list.append(torch.sigmoid(logits).cpu().numpy())
        avg_targets_list.append(yb.numpy())

avg_probs = np.concatenate(avg_probs_list)
avg_targets = np.concatenate(avg_targets_list)
avg_preds = (avg_probs >= 0.5).astype(int)

print(f"\nCheckpoint-averaged (top-{E4_TOPK}) | "
      f"Micro {f1_score(avg_targets, avg_preds, average='micro', zero_division=0):.4f} | "
      f"Macro {f1_score(avg_targets, avg_preds, average='macro', zero_division=0):.4f}")

# --- Statistically compare against Exp1/Exp2 using the bootstrap utility ---
print("\nExp4 (best checkpoint) CI:")
bootstrap_macro_f1_ci(exp4_best_probs, exp4_best_targets)

Epoch 01 | Train 0.0928 | Val 0.0948 | Micro 0.4996 | Macro 0.5374 (smoothed 0.5374) | EM 0.1484
Epoch 02 | Train 0.0700 | Val 0.0895 | Micro 0.5400 | Macro 0.5712 (smoothed 0.5543) | EM 0.1621
Epoch 03 | Train 0.0625 | Val 0.0938 | Micro 0.5168 | Macro 0.5584 (smoothed 0.5557) | EM 0.1461
Epoch 04 | Train 0.0570 | Val 0.0918 | Micro 0.5410 | Macro 0.5719 (smoothed 0.5671) | EM 0.1507
Epoch 05 | Train 0.0522 | Val 0.0999 | Micro 0.5085 | Macro 0.5471 (smoothed 0.5591) | EM 0.1530
Epoch 06 | Train 0.0522 | Val 0.0924 | Micro 0.5467 | Macro 0.5630 (smoothed 0.5607) | EM 0.1986
Epoch 07 | Train 0.0496 | Val 0.0933 | Micro 0.5451 | Macro 0.5652 (smoothed 0.5584) | EM 0.2032
Epoch 08 | Train 0.0474 | Val 0.0942 | Micro 0.5580 | Macro 0.5809 (smoothed 0.5697) | EM 0.1826
Epoch 09 | Train 0.0446 | Val 0.0932 | Micro 0.5695 | Macro 0.5863 (smoothed 0.5775) | EM 0.2237
Epoch 10 | Train 0.0451 | Val 0.0965 | Micro 0.5497 | Macro 0.5744 (smoothed 0.5806) | EM 0.1735
Epoch 11 | Train 0.0447 | Val 

(0.5969791934569346,
 np.float64(0.5500124703196132),
 np.float64(0.6391373286781312))

In [19]:
# =============================================================================
# CELL 16 — PAIRED SIGNIFICANCE TEST: EXPERIMENT 4 vs EXPERIMENT 2 (CALIBRATED)
# =============================================================================
# Step 1: calibrate Experiment 4's own thresholds (Cell 11's `thresholds`
#         belong to Experiment 2 — Exp4 needs its own).
# Step 2: paired bootstrap comparison, same resampled patients used for both,
#         so the test isolates whether Exp4 is actually better than Exp2
#         rather than picking up val-set sampling noise.

import numpy as np
from sklearn.metrics import f1_score

print("=" * 80)
print("CELL 16 — EXPERIMENT 4 CALIBRATION + PAIRED COMPARISON vs EXPERIMENT 2")
print("=" * 80)

# --- Step 1: calibrate Experiment 4 -----------------------------------------

exp4_thresholds, _ = optimize_thresholds(exp4_best_probs, exp4_best_targets, DISEASE_NAMES)

print("\n--- Experiment 4 (threshold=0.50 baseline) ---")
exp4_base_preds = (exp4_best_probs >= 0.5).astype(int)
print(f"Macro-F1: {f1_score(exp4_best_targets, exp4_base_preds, average='macro', zero_division=0):.4f}")

apply_thresholds(exp4_best_probs, exp4_best_targets, exp4_thresholds, DISEASE_NAMES, label="Experiment 4")

# --- Step 2: paired bootstrap significance test -----------------------------

def paired_bootstrap_delta(probs_a, targets_a, thresh_a, probs_b, targets_b, thresh_b,
                            label_a="A", label_b="B", n_boot=2000, seed=42):
    """
    Compares two sets of (probs, targets, thresholds) using the SAME resampled
    patient indices for both, so the test isolates the A-vs-B difference from
    val-set sampling noise. Requires probs_a/targets_a and probs_b/targets_b
    to be aligned to the same patient ordering.

    NOTE: if either threshold set was fit on this same val set (as Exp2's and
    Exp4's were), this measures consistency across the population — it does
    NOT correct for the optimism of fitting thresholds on the set you're
    scoring. A fresh held-out test set is still the right final check.
    """
    assert len(targets_a) == len(targets_b), "Mismatched patient counts — probs/targets must be aligned"

    rng = np.random.default_rng(seed)
    n = len(targets_a)

    preds_a = (probs_a >= thresh_a).astype(int)
    preds_b = (probs_b >= thresh_b).astype(int)

    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        f1_a = f1_score(targets_a[idx], preds_a[idx], average="macro", zero_division=0)
        f1_b = f1_score(targets_b[idx], preds_b[idx], average="macro", zero_division=0)
        deltas[i] = f1_b - f1_a

    lo, hi = np.percentile(deltas, [2.5, 97.5])
    p_positive = (deltas > 0).mean()

    print(f"\n--- {label_b} vs {label_a} ---")
    print(f"Mean Δ Macro-F1 ({label_b} - {label_a}): {deltas.mean():.4f}")
    print(f"95% CI of Δ: [{lo:.4f}, {hi:.4f}]")
    print(f"P({label_b} > {label_a}): {p_positive:.3f}")
    print("-> Real difference (CI excludes 0)" if (lo > 0 or hi < 0)
          else "-> NOT distinguishable from noise (CI includes 0)")

    return deltas


# best_probs / best_targets / thresholds here are Experiment 2's, from Cell 11.
# If you've since re-run Cell 11 for a different experiment, re-point these.
print(f"\nSanity check — Exp2 calibrated Macro-F1 (Cell 11 result): "
      f"{f1_score(best_targets, (best_probs >= thresholds).astype(int), average='macro', zero_division=0):.4f}")
print(f"Sanity check — Exp4 calibrated Macro-F1 (this cell):        "
      f"{f1_score(exp4_best_targets, (exp4_best_probs >= exp4_thresholds).astype(int), average='macro', zero_division=0):.4f}")

_ = paired_bootstrap_delta(
    best_probs, best_targets, thresholds,
    exp4_best_probs, exp4_best_targets, exp4_thresholds,
    label_a="Exp2 (calibrated)", label_b="Exp4 (calibrated)"
)

assert len(exp4_thresholds) == NUM_CLASSES
print("\n✓ Experiment 4 calibrated and compared against Experiment 2")

CELL 16 — EXPERIMENT 4 CALIBRATION + PAIRED COMPARISON vs EXPERIMENT 2

--- Experiment 4 (threshold=0.50 baseline) ---
Macro-F1: 0.5970

--- Experiment 4 (calibrated thresholds) ---
Micro-F1: 0.5949 | Macro-F1: 0.6259 | Exact Match: 0.3059
  N: threshold=0.51  F1=0.6036
  D: threshold=0.56  F1=0.6164
  G: threshold=0.43  F1=0.7324
  C: threshold=0.51  F1=0.8085
  A: threshold=0.79  F1=0.5000
  H: threshold=0.72  F1=0.3846
  M: threshold=0.57  F1=0.8696
  O: threshold=0.50  F1=0.4924

Sanity check — Exp2 calibrated Macro-F1 (Cell 11 result): 0.6072
Sanity check — Exp4 calibrated Macro-F1 (this cell):        0.6259

--- Exp4 (calibrated) vs Exp2 (calibrated) ---
Mean Δ Macro-F1 (Exp4 (calibrated) - Exp2 (calibrated)): 0.0181
95% CI of Δ: [-0.0122, 0.0510]
P(Exp4 (calibrated) > Exp2 (calibrated)): 0.868
-> NOT distinguishable from noise (CI includes 0)

✓ Experiment 4 calibrated and compared against Experiment 2


In [20]:
# =============================================================================
# CELL 17 — PER-CLASS WEIGHTED ASYMMETRIC LOSS (TARGETS H, A SPECIFICALLY)
# =============================================================================

class AsymmetricLossWeighted(nn.Module):

    def __init__(self, gamma_neg=2.0, gamma_pos=1.0, clip=0.05, eps=1e-8, class_weights=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.class_weights = class_weights  # tensor [num_classes] or None

    def forward(self, logits, targets):
        xs_pos = torch.sigmoid(logits)
        xs_neg = 1.0 - xs_pos

        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        xs_pos = xs_pos.clamp(self.eps, 1.0 - self.eps)
        xs_neg = xs_neg.clamp(self.eps, 1.0 - self.eps)

        loss = targets * torch.log(xs_pos) + (1.0 - targets) * torch.log(xs_neg)
        asymmetric_weight = (
            targets * (1.0 - xs_pos.detach()).pow(self.gamma_pos) +
            (1.0 - targets) * (1.0 - xs_neg.detach()).pow(self.gamma_neg)
        )
        per_class_loss = -(loss * asymmetric_weight)  # [B, 8]

        if self.class_weights is not None:
            per_class_loss = per_class_loss * self.class_weights.to(per_class_loss.device)

        return per_class_loss.mean()


class_weights = torch.ones(NUM_CLASSES)
class_weights[DISEASE_NAMES.index('H')] = 2.5   # worst, most attention-starved class
class_weights[DISEASE_NAMES.index('A')] = 1.5   # second-worst

criterion_exp4_weighted = AsymmetricLossWeighted(
    gamma_neg=2.0, gamma_pos=1.0, clip=0.05, class_weights=class_weights
)
# Drop this in as criterion_exp4 in Cell 15 for one run.

In [21]:
# =============================================================================
# CELL 17B — PER-CLASS PAIRED BOOTSTRAP (helper for Cell 18's last call)
# =============================================================================

import numpy as np
from sklearn.metrics import f1_score

def paired_bootstrap_delta_per_class(probs_a, targets_a, thresh_a, probs_b, targets_b, thresh_b,
                                      class_idx, class_name, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(targets_a)
    preds_a = (probs_a[:, class_idx] >= thresh_a[class_idx]).astype(int)
    preds_b = (probs_b[:, class_idx] >= thresh_b[class_idx]).astype(int)
    t_a, t_b = targets_a[:, class_idx], targets_b[:, class_idx]

    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        f1_a = f1_score(t_a[idx], preds_a[idx], zero_division=0)
        f1_b = f1_score(t_b[idx], preds_b[idx], zero_division=0)
        deltas[i] = f1_b - f1_a

    lo, hi = np.percentile(deltas, [2.5, 97.5])
    print(f"{class_name}: Δ F1 = {deltas.mean():.4f}, 95% CI [{lo:.4f}, {hi:.4f}]")
    print("-> Real" if (lo > 0 or hi < 0) else "-> Noise")

In [22]:
# =============================================================================
# ###EXPERIMENT 5
# CELL 18 — EXPERIMENT 5: EXP4 ARCHITECTURE + PER-CLASS WEIGHTED LOSS
# =============================================================================
# Same architecture as Experiment 4 (leaner head, no context path, checkpoint
# averaging, smoothed selection). Only change: criterion_exp4_weighted
# (from Cell 17) instead of the plain ASL. Also tracks H's own F1 every
# epoch so you can see directly whether upweighting H starts to overfit
# H specifically, not just macro.

import copy
import numpy as np
from collections import deque
from sklearn.metrics import f1_score

model_exp5 = RepresentationAwareMultiLabelHeadV4(
    input_dim=768, hidden_dim=128, num_classes=8, dropout=0.40, use_context=False
).to(DEVICE)

E5_EPOCHS = 40
E5_LR = 1e-3
E5_WEIGHT_DECAY = 1e-2
E5_PATIENCE = 8
E5_SMOOTH_WINDOW = 3
E5_TOPK = 3
H_IDX = DISEASE_NAMES.index('H')

optimizer_exp5 = torch.optim.AdamW(model_exp5.parameters(), lr=E5_LR, weight_decay=E5_WEIGHT_DECAY)

exp5_history = []
recent_macro = deque(maxlen=E5_SMOOTH_WINDOW)
top_checkpoints = []
best_smoothed_f1 = -1.0
patience_counter = 0

for epoch in range(1, E5_EPOCHS + 1):

    model_exp5.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_exp5.zero_grad()
        logits = model_exp5(xb)
        loss = criterion_exp4_weighted(logits, yb)
        loss.backward()
        optimizer_exp5.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model_exp5.eval()
    val_loss = 0.0
    probs_list, targets_list = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model_exp5(xb)
            loss = criterion_exp4_weighted(logits, yb)
            val_loss += loss.item() * xb.size(0)
            probs_list.append(torch.sigmoid(logits).cpu().numpy())
            targets_list.append(yb.cpu().numpy())
    val_loss /= len(val_loader.dataset)

    probs = np.concatenate(probs_list)
    targets = np.concatenate(targets_list)
    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(targets, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    exact_match = np.mean(np.all(preds == targets, axis=1))
    per_class_f1 = f1_score(targets, preds, average=None, zero_division=0)

    train_preds_h = (torch.sigmoid(model_exp5(X_train.to(DEVICE))[:, H_IDX]) >= 0.5).int().cpu().numpy()
    train_f1_h = f1_score(Y_train[:, H_IDX].numpy(), train_preds_h, zero_division=0)
    val_f1_h = per_class_f1[H_IDX]

    recent_macro.append(macro_f1)
    smoothed_macro_f1 = float(np.mean(recent_macro))

    exp5_history.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "micro_f1": micro_f1, "macro_f1": macro_f1,
        "smoothed_macro_f1": smoothed_macro_f1, "exact_match": exact_match,
        "per_class_f1": per_class_f1.copy(),
        "train_f1_h": train_f1_h, "val_f1_h": val_f1_h
    })

    print(
        f"Epoch {epoch:02d} | Train {train_loss:.4f} | Val {val_loss:.4f} | "
        f"Micro {micro_f1:.4f} | Macro {macro_f1:.4f} (smoothed {smoothed_macro_f1:.4f}) | "
        f"EM {exact_match:.4f} | H: train_F1 {train_f1_h:.4f} / val_F1 {val_f1_h:.4f}"
    )

    top_checkpoints.append((smoothed_macro_f1, copy.deepcopy(model_exp5.state_dict())))
    top_checkpoints = sorted(top_checkpoints, key=lambda t: t[0], reverse=True)[:E5_TOPK]

    if smoothed_macro_f1 > best_smoothed_f1:
        best_smoothed_f1 = smoothed_macro_f1
        exp5_best_epoch = epoch
        exp5_best_state = copy.deepcopy(model_exp5.state_dict())
        exp5_best_probs = probs.copy()
        exp5_best_targets = targets.copy()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= E5_PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

model_exp5.load_state_dict(exp5_best_state)
print(f"\nBest epoch (smoothed selection): {exp5_best_epoch}")
print(f"Best smoothed Macro-F1: {best_smoothed_f1:.4f}")
print(f"H — train F1: {exp5_history[exp5_best_epoch-1]['train_f1_h']:.4f} | "
      f"val F1: {exp5_history[exp5_best_epoch-1]['val_f1_h']:.4f}")

# --- Calibrate and compare against Exp4 (the real baseline now) ---
exp5_thresholds, _ = optimize_thresholds(exp5_best_probs, exp5_best_targets, DISEASE_NAMES)
apply_thresholds(exp5_best_probs, exp5_best_targets, exp5_thresholds, DISEASE_NAMES, label="Experiment 5")

_ = paired_bootstrap_delta(
    exp4_best_probs, exp4_best_targets, exp4_thresholds,
    exp5_best_probs, exp5_best_targets, exp5_thresholds,
    label_a="Exp4 (calibrated)", label_b="Exp5 (calibrated)"
)

paired_bootstrap_delta_per_class(
    exp4_best_probs, exp4_best_targets, exp4_thresholds,
    exp5_best_probs, exp5_best_targets, exp5_thresholds,
    class_idx=H_IDX, class_name="H"
)

Epoch 01 | Train 0.1266 | Val 0.1218 | Micro 0.4681 | Macro 0.5106 (smoothed 0.5106) | EM 0.1644 | H: train_F1 0.1773 / val_F1 0.1615
Epoch 02 | Train 0.0840 | Val 0.1123 | Micro 0.5056 | Macro 0.5445 (smoothed 0.5275) | EM 0.1621 | H: train_F1 0.3034 / val_F1 0.1600
Epoch 03 | Train 0.0731 | Val 0.1109 | Micro 0.5285 | Macro 0.5547 (smoothed 0.5366) | EM 0.1553 | H: train_F1 0.3467 / val_F1 0.1928
Epoch 04 | Train 0.0691 | Val 0.1106 | Micro 0.5307 | Macro 0.5646 (smoothed 0.5546) | EM 0.2306 | H: train_F1 0.4177 / val_F1 0.2029
Epoch 05 | Train 0.0643 | Val 0.1098 | Micro 0.5383 | Macro 0.5729 (smoothed 0.5641) | EM 0.1644 | H: train_F1 0.4765 / val_F1 0.2712
Epoch 06 | Train 0.0606 | Val 0.1101 | Micro 0.5403 | Macro 0.5743 (smoothed 0.5706) | EM 0.1849 | H: train_F1 0.4844 / val_F1 0.2759
Epoch 07 | Train 0.0581 | Val 0.1126 | Micro 0.5309 | Macro 0.5639 (smoothed 0.5704) | EM 0.1598 | H: train_F1 0.5763 / val_F1 0.2642
Epoch 08 | Train 0.0552 | Val 0.1094 | Micro 0.5542 | Macro 0.

In [23]:
# =============================================================================
# CELL 19 — FINAL MODEL LOCKED: EXPERIMENT 4
# =============================================================================
# Experiment 4 is the final Module 9 model:
#   - Statistically tied with Exp2, 8x fewer params, cleaner train/val gap
#   - Exp5's targeted H/A reweighting did not beat it (CI included 0)
# Locks the single best-epoch checkpoint (not the checkpoint-averaged
# variant) since that's what exp4_best_probs / exp4_thresholds / every
# paired bootstrap comparison in this notebook actually used.

import os
import json
import torch
import numpy as np
from sklearn.metrics import f1_score

print("=" * 80)
print("CELL 19 — LOCK EXPERIMENT 4 AS FINAL MODULE 9 MODEL")
print("=" * 80)

# Derives from FUSED_ARTIFACT_PATH so it lands next to your other saved
# artifacts. If that variable isn't in this session, set LOCK_DIR manually.
LOCK_DIR = os.path.join(os.path.dirname(FUSED_ARTIFACT_PATH), "module9_final")
os.makedirs(LOCK_DIR, exist_ok=True)

# --- Reload the exact checkpoint being locked, verify it's the tested one ---
model_exp4.load_state_dict(exp4_best_state)
model_exp4.eval()

verify_macro = f1_score(exp4_best_targets, (exp4_best_probs >= 0.5).astype(int),
                         average="macro", zero_division=0)
assert abs(verify_macro - 0.5970) < 1e-3, \
    f"exp4_best_probs macro-F1 ({verify_macro:.4f}) doesn't match the tested run (0.5970) " \
    "— check kernel state before locking."

# --- Assemble the model card ---
calibrated_preds = (exp4_best_probs >= exp4_thresholds).astype(int)
uncalibrated_preds = (exp4_best_probs >= 0.5).astype(int)

final_model_card = {
    "module": "Module 9 — Multi-label classification head",
    "experiment": "Experiment 4",
    "architecture": {
        "class": "RepresentationAwareMultiLabelHeadV4",
        "input_dim": 768, "hidden_dim": 128,
        "num_classes": NUM_CLASSES, "dropout": 0.40, "use_context": False,
    },
    "training": {
        "loss": "AsymmetricLoss(gamma_neg=2.0, gamma_pos=1.0, clip=0.05)",
        "optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-2,
        "patience": 8, "smooth_window": 3,
        "best_epoch": exp4_best_epoch,
        "best_smoothed_macro_f1": float(best_smoothed_f1),
    },
    "thresholds": {name: float(t) for name, t in zip(DISEASE_NAMES, exp4_thresholds)},
    "metrics_uncalibrated": {
        "macro_f1": float(f1_score(exp4_best_targets, uncalibrated_preds, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(exp4_best_targets, uncalibrated_preds, average="micro", zero_division=0)),
    },
    "metrics_calibrated": {
        "macro_f1": float(f1_score(exp4_best_targets, calibrated_preds, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(exp4_best_targets, calibrated_preds, average="micro", zero_division=0)),
        "per_class_f1": {
            name: float(f1) for name, f1 in zip(
                DISEASE_NAMES,
                f1_score(exp4_best_targets, calibrated_preds, average=None, zero_division=0)
            )
        },
    },
    "comparisons": {
        "vs_experiment_2": "CI [-0.0122, 0.0510], not distinguishable — adopted for 8x fewer params + cleaner train/val gap",
        "vs_experiment_5": "H/A-reweighted loss did not beat Exp4 — macro CI [-0.0394, 0.0097], H delta CI [-0.1629, 0.1118]",
    },
    "known_limitation": "Hypertension (H) F1 ~0.38 — traced to Module 6 disease-attention entropy "
                         "(0.9168 of max, near-uniform routing), not fixable within Module 9's frozen-representation scope.",
}

# --- Save weights, thresholds, and model card ---
weights_path = os.path.join(LOCK_DIR, "module9_exp4_final.pt")
thresholds_path = os.path.join(LOCK_DIR, "module9_exp4_thresholds.npy")
card_path = os.path.join(LOCK_DIR, "module9_exp4_model_card.json")

torch.save(model_exp4.state_dict(), weights_path)
np.save(thresholds_path, exp4_thresholds)
with open(card_path, "w") as f:
    json.dump(final_model_card, f, indent=2)

# --- Round-trip verification: reload from disk, confirm predictions match exactly ---
reload_model = RepresentationAwareMultiLabelHeadV4(
    input_dim=768, hidden_dim=128, num_classes=NUM_CLASSES, dropout=0.40, use_context=False
).to(DEVICE)
reload_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
reload_model.eval()

with torch.no_grad():
    reload_probs = np.concatenate([
        torch.sigmoid(reload_model(xb.to(DEVICE))).cpu().numpy()
        for xb, _ in val_loader
    ])

assert np.allclose(reload_probs, exp4_best_probs, atol=1e-5), \
    "Reloaded predictions don't match the locked checkpoint — save/load mismatch."
assert np.allclose(np.load(thresholds_path), exp4_thresholds)

print(f"\n✓ Weights saved:    {weights_path}")
print(f"✓ Thresholds saved: {thresholds_path}")
print(f"✓ Model card saved: {card_path}")
print(f"✓ Round-trip verified — reloaded model reproduces exp4_best_probs exactly")
print(f"\nFinal Module 9 model: Experiment 4, epoch {exp4_best_epoch}")
print(f"Calibrated Macro-F1: {final_model_card['metrics_calibrated']['macro_f1']:.4f}")
print(f"Calibrated Micro-F1: {final_model_card['metrics_calibrated']['micro_f1']:.4f}")

CELL 19 — LOCK EXPERIMENT 4 AS FINAL MODULE 9 MODEL

✓ Weights saved:    /content/drive/My Drive/Eye Disease/Dataset/module9_final/module9_exp4_final.pt
✓ Thresholds saved: /content/drive/My Drive/Eye Disease/Dataset/module9_final/module9_exp4_thresholds.npy
✓ Model card saved: /content/drive/My Drive/Eye Disease/Dataset/module9_final/module9_exp4_model_card.json
✓ Round-trip verified — reloaded model reproduces exp4_best_probs exactly

Final Module 9 model: Experiment 4, epoch 19
Calibrated Macro-F1: 0.6259
Calibrated Micro-F1: 0.5949


###EVALUATION

In [24]:
# =============================================================================
# CELL 20 — LOAD HELD-OUT TEST SPLIT
# =============================================================================

TEST_DF_PATH = ROOT / "test_patient_df.csv"

assert TEST_DF_PATH.exists(), f"Missing test file: {TEST_DF_PATH}"

test_patient_df = pd.read_csv(TEST_DF_PATH)

print("=" * 80)
print("CELL 20 — HELD-OUT TEST SPLIT")
print("=" * 80)

print(f"Test shape: {test_patient_df.shape}")
print("\nColumns:")
print(test_patient_df.columns.tolist())

print("\nFirst 5 rows:")
display(test_patient_df.head())

CELL 20 — HELD-OUT TEST SPLIT
Test shape: (504, 11)

Columns:
['patient_id', 'right_filename', 'left_filename', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

First 5 rows:


,patient_id,right_filename,left_filename,N,D,G,C,A,H,M,O
0,0,0_right.jpg,0_left.jpg,0,0,0,1,0,0,0,0
1,15,15_right.jpg,15_left.jpg,0,0,0,0,0,0,0,1
2,33,33_right.jpg,33_left.jpg,0,0,0,0,0,0,0,1
3,47,47_right.jpg,47_left.jpg,0,1,0,0,0,0,0,1
4,49,49_right.jpg,49_left.jpg,0,1,0,0,0,0,0,0


In [25]:
# =============================================================================
# CELL 21 — VERIFY BILATERAL TEST COHORT
# =============================================================================

TEST_IMAGE_DIR = ROOT / "preprocessed_images"

def file_exists(filename):
    if pd.isna(filename):
        return False
    return (TEST_IMAGE_DIR / str(filename)).exists()

test_patient_df["right_exists"] = test_patient_df["right_filename"].apply(file_exists)
test_patient_df["left_exists"] = test_patient_df["left_filename"].apply(file_exists)

test_bilateral_df = test_patient_df[
    test_patient_df["right_exists"] & test_patient_df["left_exists"]
].copy()

print("=" * 80)
print("CELL 21 — BILATERAL TEST COHORT")
print("=" * 80)

print(f"Original test patients : {len(test_patient_df)}")
print(f"Bilateral test patients: {len(test_bilateral_df)}")
print(f"Excluded patients      : {len(test_patient_df) - len(test_bilateral_df)}")

print("\nMissing right-eye files:", (~test_patient_df["right_exists"]).sum())
print("Missing left-eye files :", (~test_patient_df["left_exists"]).sum())

assert len(test_bilateral_df) > 0
assert test_bilateral_df["patient_id"].is_unique

print("\n✓ Bilateral test cohort created.")

CELL 21 — BILATERAL TEST COHORT
Original test patients : 504
Bilateral test patients: 455
Excluded patients      : 49

Missing right-eye files: 30
Missing left-eye files : 19

✓ Bilateral test cohort created.


In [26]:
# =============================================================================
# CELL 22 — PREPARE BILATERAL TEST EYE RECORDS
# =============================================================================

PREPROCESS_DIR = ROOT / "preprocessed_images"

right_test_df = test_bilateral_df[
    ["patient_id", "right_filename"] + DISEASE_NAMES
].copy()

right_test_df = right_test_df.rename(
    columns={"right_filename": "filename"}
)
right_test_df["eye"] = "right"

left_test_df = test_bilateral_df[
    ["patient_id", "left_filename"] + DISEASE_NAMES
].copy()

left_test_df = left_test_df.rename(
    columns={"left_filename": "filename"}
)
left_test_df["eye"] = "left"

bilateral_test_eye_df = pd.concat(
    [right_test_df, left_test_df],
    ignore_index=True
)

bilateral_test_eye_df["image_path"] = (
    bilateral_test_eye_df["filename"]
    .apply(lambda x: str(PREPROCESS_DIR / str(x)))
)

print("=" * 80)
print("CELL 22 — BILATERAL TEST EYE RECORDS")
print("=" * 80)

print("Bilateral test patients:", len(test_bilateral_df))
print("Test eye records:", len(bilateral_test_eye_df))

print(
    "Right eyes:",
    (bilateral_test_eye_df["eye"] == "right").sum()
)

print(
    "Left eyes:",
    (bilateral_test_eye_df["eye"] == "left").sum()
)

print(
    "Existing image files:",
    bilateral_test_eye_df["image_path"].apply(os.path.exists).sum()
)

assert len(bilateral_test_eye_df) == 455 * 2
assert bilateral_test_eye_df["patient_id"].nunique() == 455
assert bilateral_test_eye_df["image_path"].apply(os.path.exists).all()

print("\n✓ Bilateral test eye records ready.")
print("✓ All 910 test eye images found.")

CELL 22 — BILATERAL TEST EYE RECORDS
Bilateral test patients: 455
Test eye records: 910
Right eyes: 455
Left eyes: 455
Existing image files: 910

✓ Bilateral test eye records ready.
✓ All 910 test eye images found.


In [27]:
# =============================================================================
# CELL 23 — CHECK TEST QUALITY SCORES
# =============================================================================

TEST_QUALITY_PATH = ROOT / "test_quality_scores.csv"

print("=" * 80)
print("CELL 23 — TEST QUALITY SCORE CHECK")
print("=" * 80)

print("Expected file:")
print(TEST_QUALITY_PATH)

print("Exists:", TEST_QUALITY_PATH.exists())

if TEST_QUALITY_PATH.exists():
    test_quality_scores_df = pd.read_csv(TEST_QUALITY_PATH)

    print("Rows:", len(test_quality_scores_df))
    print("Columns:", test_quality_scores_df.columns.tolist())
else:
    print("\n⚠ test_quality_scores.csv was not found.")

CELL 23 — TEST QUALITY SCORE CHECK
Expected file:
/content/drive/My Drive/Eye Disease/Dataset/test_quality_scores.csv
Exists: True
Rows: 910
Columns: ['patient_id', 'eye', 'filename', 'reference_deviation_score', 'normalized_deviation', 'quality_score']


In [29]:
# =============================================================================
# CELL 24 — GENERATE FROZEN TEST QUALITY SCORES
# =============================================================================

import cv2
from skimage.measure import shannon_entropy
from sklearn.preprocessing import MinMaxScaler

QUALITY_COLUMNS = ["blur", "brightness", "contrast", "entropy"]

TRAIN_QUALITY_PATH = ROOT / "train_quality_metrics.csv"
TEST_QUALITY_PATH = ROOT / "test_quality_scores.csv"

assert TRAIN_QUALITY_PATH.exists(), "Training quality metrics not found."

train_raw = pd.read_csv(TRAIN_QUALITY_PATH)

def calculate_quality_metrics(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return {
        "blur": cv2.Laplacian(gray, cv2.CV_64F).var(),
        "brightness": np.mean(gray),
        "contrast": np.std(gray),
        "entropy": shannon_entropy(gray)
    }

# -------------------------------------------------------------------------
# 1. Fit metric scaler ONLY on training data
# -------------------------------------------------------------------------

metric_scaler = MinMaxScaler()
metric_scaler.fit(train_raw[QUALITY_COLUMNS])

train_norm = train_raw.copy()
train_norm[QUALITY_COLUMNS] = metric_scaler.transform(
    train_raw[QUALITY_COLUMNS]
)

# -------------------------------------------------------------------------
# 2. Training clean-reference statistics
# -------------------------------------------------------------------------

reference_mean = train_norm[QUALITY_COLUMNS].mean()
reference_std = train_norm[QUALITY_COLUMNS].std()

# -------------------------------------------------------------------------
# 3. Training reference-deviation scores
# -------------------------------------------------------------------------

train_deviation = pd.DataFrame(index=train_norm.index)

for metric in QUALITY_COLUMNS:
    train_deviation[f"{metric}_deviation"] = (
        (train_norm[metric] - reference_mean[metric]).abs()
        / reference_std[metric]
    )

train_deviation["reference_deviation_score"] = (
    train_deviation.mean(axis=1)
)

# -------------------------------------------------------------------------
# 4. Fit final quality scaler ONLY on training deviation scores
# -------------------------------------------------------------------------

quality_scaler = MinMaxScaler()

quality_scaler.fit(
    train_deviation[["reference_deviation_score"]]
)

# -------------------------------------------------------------------------
# 5. Calculate raw metrics for test eyes
# -------------------------------------------------------------------------

records = []

for _, row in bilateral_test_eye_df.iterrows():

    image = cv2.imread(row["image_path"])

    if image is None:
        raise FileNotFoundError(row["image_path"])

    metrics = calculate_quality_metrics(image)

    records.append({
        "patient_id": row["patient_id"],
        "eye": row["eye"],
        "filename": row["filename"],
        **metrics
    })

test_quality = pd.DataFrame(records)

# -------------------------------------------------------------------------
# 6. Apply TRAINING scaler to test metrics
# -------------------------------------------------------------------------

test_norm = test_quality.copy()

test_norm[QUALITY_COLUMNS] = metric_scaler.transform(
    test_quality[QUALITY_COLUMNS]
)

# -------------------------------------------------------------------------
# 7. Calculate test reference deviations
# -------------------------------------------------------------------------

for metric in QUALITY_COLUMNS:
    test_norm[f"{metric}_deviation"] = (
        (test_norm[metric] - reference_mean[metric]).abs()
        / reference_std[metric]
    )

deviation_columns = [
    f"{metric}_deviation"
    for metric in QUALITY_COLUMNS
]

test_norm["reference_deviation_score"] = (
    test_norm[deviation_columns].mean(axis=1)
)

# -------------------------------------------------------------------------
# 8. Apply TRAINING quality scaler and invert
# -------------------------------------------------------------------------

test_norm["normalized_deviation"] = quality_scaler.transform(
    test_norm[["reference_deviation_score"]]
).flatten()

test_norm["quality_score"] = (
    1 - test_norm["normalized_deviation"]
)

# -------------------------------------------------------------------------
# 9. Save
# -------------------------------------------------------------------------

test_quality_scores_df = test_norm[
    [
        "patient_id",
        "eye",
        "filename",
        "reference_deviation_score",
        "normalized_deviation",
        "quality_score"
    ]
].copy()

test_quality_scores_df.to_csv(
    TEST_QUALITY_PATH,
    index=False
)

print("=" * 80)
print("CELL 24 — TEST QUALITY SCORES GENERATED")
print("=" * 80)

print("Test eyes:", len(test_quality_scores_df))
print("Unique patients:", test_quality_scores_df["patient_id"].nunique())
print(
    "Quality score range:",
    test_quality_scores_df["quality_score"].min(),
    "to",
    test_quality_scores_df["quality_score"].max()
)

print("Saved:", TEST_QUALITY_PATH)

assert len(test_quality_scores_df) == 910
assert test_quality_scores_df["filename"].nunique() == 910
assert test_quality_scores_df["quality_score"].notna().all()

print("\n✓ 910 test quality scores generated.")
print("✓ No scaler or reference statistic was fitted on test data.")

CELL 24 — TEST QUALITY SCORES GENERATED
Test eyes: 910
Unique patients: 455
Quality score range: 0.22075683845199645 to 0.9977300912004421
Saved: /content/drive/My Drive/Eye Disease/Dataset/test_quality_scores.csv

✓ 910 test quality scores generated.
✓ No scaler or reference statistic was fitted on test data.


In [30]:
# =============================================================================
# CELL 25 — FROZEN QUALITY-FiLM TEST FEATURE EXTRACTION
# =============================================================================

import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

CHECKPOINT_PATH = ROOT / "quality_conditioned_convnext_checkpoint.pt"

assert CHECKPOINT_PATH.exists()

IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class QualityEmbedding(nn.Module):
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(1, 32),
            nn.GELU(),
            nn.Linear(32, embedding_dim),
            nn.GELU()
        )

    def forward(self, quality_score):
        if quality_score.dim() == 1:
            quality_score = quality_score.unsqueeze(1)
        return self.network(quality_score)


class FiLMGenerator(nn.Module):
    def __init__(self, quality_embedding_dim=64, feature_dim=768):
        super().__init__()
        self.generator = nn.Sequential(
            nn.Linear(quality_embedding_dim, 128),
            nn.GELU(),
            nn.Linear(128, feature_dim * 2)
        )

    def forward(self, quality_embedding):
        return torch.chunk(
            self.generator(quality_embedding),
            2,
            dim=1
        )


class QualityConditionedConvNeXt(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.quality_embedding = QualityEmbedding(64)
        self.film_generator = FiLMGenerator(64, 768)

    def forward(self, image, quality_score):
        features = torch.flatten(
            self.backbone(image),
            start_dim=1
        )

        quality_embedding = self.quality_embedding(
            quality_score
        )

        gamma_raw, beta = self.film_generator(
            quality_embedding
        )

        return (1.0 + gamma_raw) * features + beta


from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

backbone = convnext_tiny(
    weights=ConvNeXt_Tiny_Weights.DEFAULT
)

backbone.classifier = nn.Identity()

quality_model = QualityConditionedConvNeXt(
    backbone
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

quality_model.load_state_dict(
    {
        k.replace("feature_extractor.", ""): v
        for k, v in checkpoint["model_state_dict"].items()
        if k.startswith("feature_extractor.")
    }
)

quality_model.eval()

quality_lookup = (
    test_quality_scores_df
    .set_index(["patient_id", "eye"])
)

test_features = []
test_ids = []
test_eyes = []

for _, row in tqdm(
    bilateral_test_eye_df.iterrows(),
    total=len(bilateral_test_eye_df),
    desc="Extracting Quality-FiLM features"
):

    image = Image.open(
        row["image_path"]
    ).convert("RGB")

    image = IMAGE_TRANSFORM(image).unsqueeze(0).to(DEVICE)

    quality_score = torch.tensor(
        [quality_lookup.loc[
            (row["patient_id"], row["eye"]),
            "quality_score"
        ]],
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():
        features = quality_model(
            image,
            quality_score
        )

    test_features.append(
        features.cpu()
    )

    test_ids.append(row["patient_id"])
    test_eyes.append(row["eye"])


test_features = torch.cat(test_features, dim=0)

print("=" * 80)
print("CELL 25 — QUALITY-FiLM TEST FEATURES")
print("=" * 80)

print("Feature shape:", test_features.shape)
print("Patients:", len(set(test_ids)))
print("Eyes:", len(test_ids))

assert test_features.shape == (910, 768)
assert torch.isfinite(test_features).all()

print("\n✓ Frozen Quality-FiLM model restored.")
print("✓ 910 test eyes processed.")
print("✓ Feature dimension: 768")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 185MB/s] 


Extracting Quality-FiLM features:   0%|          | 0/910 [00:00<?, ?it/s]

CELL 25 — QUALITY-FiLM TEST FEATURES
Feature shape: torch.Size([910, 768])
Patients: 455
Eyes: 910

✓ Frozen Quality-FiLM model restored.
✓ 910 test eyes processed.
✓ Feature dimension: 768


In [31]:
# =============================================================================
# CELL 26 — FROZEN DISEASE-AWARE ATTENTION
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

DAA_CHECKPOINT = ROOT / "disease_aware_attention_best.pt"
PROTOTYPES_PATH = ROOT / "disease_prototypes.pt"

assert DAA_CHECKPOINT.exists()
assert PROTOTYPES_PATH.exists()


class DiseaseAwareAttention(nn.Module):
    def __init__(self, feature_dim=768, prototype_dim=256):
        super().__init__()

        self.query_projection = nn.Sequential(
            nn.Linear(feature_dim, prototype_dim),
            nn.LayerNorm(prototype_dim),
            nn.GELU(),
            nn.Linear(prototype_dim, prototype_dim)
        )

        self.key_projection = nn.Linear(
            feature_dim, prototype_dim
        )

        self.value_projection = nn.Linear(
            feature_dim, prototype_dim
        )

        self.output_projection = nn.Linear(
            prototype_dim, feature_dim
        )

        self.norm = nn.LayerNorm(feature_dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x, prototypes):

        q = self.query_projection(x)
        k = self.key_projection(prototypes)
        v = self.value_projection(prototypes)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        scores = torch.matmul(q, k.T)

        attention = torch.softmax(
            scores,
            dim=-1
        )

        attended = torch.matmul(
            attention,
            v
        )

        attended = self.output_projection(
            attended
        )

        attended = self.dropout(
            attended
        )

        output = self.norm(
            x + attended
        )

        return output, attention


class DiseaseAwareAttentionClassifier(nn.Module):
    def __init__(
        self,
        feature_dim=768,
        prototype_dim=256,
        num_classes=8
    ):
        super().__init__()

        self.attention = DiseaseAwareAttention(
            feature_dim,
            prototype_dim
        )

        self.classifier = nn.Linear(
            feature_dim,
            num_classes
        )

    def forward(self, x, prototypes):

        features, attention = self.attention(
            x,
            prototypes
        )

        logits = self.classifier(
            features
        )

        return features, attention, logits


prototypes = torch.load(
    PROTOTYPES_PATH,
    map_location=DEVICE
)

if isinstance(prototypes, dict):
    prototypes = prototypes["disease_prototypes"]

prototypes = prototypes.float().to(DEVICE)


daa_model = DiseaseAwareAttentionClassifier(
    feature_dim=768,
    prototype_dim=256,
    num_classes=8
).to(DEVICE)

checkpoint = torch.load(
    DAA_CHECKPOINT,
    map_location=DEVICE
)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint
)

daa_model.load_state_dict(
    state_dict
)

daa_model.eval()


test_daa_features = []
test_daa_attention = []

BATCH_SIZE_DAA = 64

with torch.no_grad():

    for start in range(
        0,
        len(test_features),
        BATCH_SIZE_DAA
    ):

        xb = test_features[
            start:start + BATCH_SIZE_DAA
        ].to(DEVICE)

        features, attention, _ = daa_model(
            xb,
            prototypes
        )

        test_daa_features.append(
            features.cpu()
        )

        test_daa_attention.append(
            attention.cpu()
        )


test_daa_features = torch.cat(
    test_daa_features,
    dim=0
)

test_daa_attention = torch.cat(
    test_daa_attention,
    dim=0
)


print("=" * 80)
print("CELL 26 — DISEASE-AWARE ATTENTION TEST FEATURES")
print("=" * 80)

print(
    "DAA feature shape:",
    test_daa_features.shape
)

print(
    "Attention shape:",
    test_daa_attention.shape
)

assert test_daa_features.shape == (910, 768)
assert test_daa_attention.shape == (910, 8)

assert torch.isfinite(
    test_daa_features
).all()

assert torch.isfinite(
    test_daa_attention
).all()

print("\n✓ Exact frozen DAA architecture restored.")
print("✓ Frozen DAA checkpoint loaded.")
print("✓ Disease prototypes loaded.")
print("✓ 910 test eyes processed.")

CELL 26 — DISEASE-AWARE ATTENTION TEST FEATURES
DAA feature shape: torch.Size([910, 768])
Attention shape: torch.Size([910, 8])

✓ Exact frozen DAA architecture restored.
✓ Frozen DAA checkpoint loaded.
✓ Disease prototypes loaded.
✓ 910 test eyes processed.


In [32]:
# =============================================================================
# CELL 27 — PREPARE PAIRED TEST EYE REPRESENTATIONS
# =============================================================================

test_daa_df = bilateral_test_eye_df[
    ["patient_id", "eye", "filename"]
].copy()

test_daa_df["row_idx"] = np.arange(
    len(test_daa_df)
)

right_idx = (
    test_daa_df["eye"] == "right"
)

left_idx = (
    test_daa_df["eye"] == "left"
)

right_df = test_daa_df[right_idx].copy()
left_df = test_daa_df[left_idx].copy()

right_df = right_df.sort_values(
    "patient_id"
).reset_index(drop=True)

left_df = left_df.sort_values(
    "patient_id"
).reset_index(drop=True)

assert np.array_equal(
    right_df["patient_id"].values,
    left_df["patient_id"].values
)

right_rows = right_df["row_idx"].values
left_rows = left_df["row_idx"].values

right_daa = test_daa_features[right_rows]
left_daa = test_daa_features[left_rows]

right_quality = torch.tensor(
    test_quality_scores_df[
        test_quality_scores_df["eye"] == "right"
    ].sort_values("patient_id")["quality_score"].values,
    dtype=torch.float32
)

left_quality = torch.tensor(
    test_quality_scores_df[
        test_quality_scores_df["eye"] == "left"
    ].sort_values("patient_id")["quality_score"].values,
    dtype=torch.float32
)

print("=" * 80)
print("CELL 27 — PAIRED TEST REPRESENTATIONS")
print("=" * 80)

print("Patients:", len(right_df))
print("Right DAA:", right_daa.shape)
print("Left DAA:", left_daa.shape)
print("Right quality:", right_quality.shape)
print("Left quality:", left_quality.shape)

assert right_daa.shape == (455, 768)
assert left_daa.shape == (455, 768)
assert right_quality.shape == (455,)
assert left_quality.shape == (455,)

print("\n✓ Right/left patient pairing verified.")
print("✓ 455 bilateral test patients ready.")

CELL 27 — PAIRED TEST REPRESENTATIONS
Patients: 455
Right DAA: torch.Size([455, 768])
Left DAA: torch.Size([455, 768])
Right quality: torch.Size([455])
Left quality: torch.Size([455])

✓ Right/left patient pairing verified.
✓ 455 bilateral test patients ready.


In [33]:
# =============================================================================
# CELL 28 — RESTORE FROZEN CROSS-EYE MODEL
# =============================================================================

class CrossEyeBilateralAttention(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_tokens=8,
        token_dim=96,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.feature_dim = feature_dim
        self.num_tokens = num_tokens
        self.token_dim = token_dim

        self.tokenizer = nn.Sequential(
            nn.Linear(feature_dim, num_tokens * token_dim),
            nn.LayerNorm(num_tokens * token_dim),
            nn.GELU()
        )

        self.right_queries_left = nn.MultiheadAttention(
            embed_dim=token_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.left_queries_right = nn.MultiheadAttention(
            embed_dim=token_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.right_token_norm = nn.LayerNorm(token_dim)
        self.left_token_norm = nn.LayerNorm(token_dim)

        self.right_output_projection = nn.Sequential(
            nn.Linear(num_tokens * token_dim, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.left_output_projection = nn.Sequential(
            nn.Linear(num_tokens * token_dim, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.right_feature_norm = nn.LayerNorm(feature_dim)
        self.left_feature_norm = nn.LayerNorm(feature_dim)

        self.right_gate = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.Sigmoid()
        )

        self.left_gate = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.Sigmoid()
        )

    def _tokenize(self, features):
        batch_size = features.size(0)

        tokens = self.tokenizer(features)

        return tokens.view(
            batch_size,
            self.num_tokens,
            self.token_dim
        )

    def forward(self, right_features, left_features):

        right_tokens = self._tokenize(right_features)
        left_tokens = self._tokenize(left_features)

        right_context, right_attention = self.right_queries_left(
            query=right_tokens,
            key=left_tokens,
            value=left_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        left_context, left_attention = self.left_queries_right(
            query=left_tokens,
            key=right_tokens,
            value=right_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        right_context = self.right_token_norm(
            right_tokens + right_context
        )

        left_context = self.left_token_norm(
            left_tokens + left_context
        )

        right_cross_eye = self.right_output_projection(
            right_context.reshape(right_context.size(0), -1)
        )

        left_cross_eye = self.left_output_projection(
            left_context.reshape(left_context.size(0), -1)
        )

        right_gate = self.right_gate(
            torch.cat([right_features, right_cross_eye], dim=1)
        )

        left_gate = self.left_gate(
            torch.cat([left_features, left_cross_eye], dim=1)
        )

        right_output = self.right_feature_norm(
            right_features + right_gate * right_cross_eye
        )

        left_output = self.left_feature_norm(
            left_features + left_gate * left_cross_eye
        )

        return {
            "right_output": right_output,
            "left_output": left_output,
            "right_attention": right_attention,
            "left_attention": left_attention
        }


class CrossEyeTrainingModel(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_tokens=8,
        token_dim=96,
        num_heads=4,
        num_classes=8,
        dropout=0.1
    ):
        super().__init__()

        self.cross_eye_attention = CrossEyeBilateralAttention(
            feature_dim=feature_dim,
            num_tokens=num_tokens,
            token_dim=token_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.patient_projection = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, right_features, left_features):

        bilateral = self.cross_eye_attention(
            right_features,
            left_features
        )

        right_output = bilateral["right_output"]
        left_output = bilateral["left_output"]

        patient_features = self.patient_projection(
            torch.cat(
                [right_output, left_output],
                dim=1
            )
        )

        logits = self.classifier(patient_features)

        return {
            "right_features": right_output,
            "left_features": left_output,
            "patient_features": patient_features,
            "logits": logits,
            "right_cross_attention": bilateral["right_attention"],
            "left_cross_attention": bilateral["left_attention"]
        }


CROSS_EYE_CHECKPOINT = ROOT / "cross_eye_bilateral_attention_best.pt"

assert CROSS_EYE_CHECKPOINT.exists(), (
    f"Missing Cross-Eye checkpoint: {CROSS_EYE_CHECKPOINT}"
)

cross_eye_model = CrossEyeTrainingModel(
    feature_dim=768,
    num_tokens=8,
    token_dim=96,
    num_heads=4,
    num_classes=8,
    dropout=0.1
).to(DEVICE)

checkpoint = torch.load(
    CROSS_EYE_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

cross_eye_model.load_state_dict(
    checkpoint["model_state_dict"]
)

cross_eye_model.eval()

print("=" * 80)
print("CELL 28 — FROZEN CROSS-EYE MODEL")
print("=" * 80)
print("Checkpoint:", CROSS_EYE_CHECKPOINT)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation loss:", f"{checkpoint['validation_loss']:.6f}")
print("✓ Exact Cross-Eye architecture restored.")
print("✓ Frozen checkpoint loaded.")

CELL 28 — FROZEN CROSS-EYE MODEL
Checkpoint: /content/drive/My Drive/Eye Disease/Dataset/cross_eye_bilateral_attention_best.pt
Checkpoint epoch: 1
Validation loss: 0.101313
✓ Exact Cross-Eye architecture restored.
✓ Frozen checkpoint loaded.
